In [1]:
# Cell 1 — Setup, imports, Google Drive

import os
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.preprocessing import MinMaxScaler

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Không chạy trong Google Colab; bỏ qua bước mount Drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")


Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:
# Cell 2 — PAPER MODE configuration

# Paper output is deliberately separated from pilot output.
EXPERIMENT_TAG = "conference_step4_exact_feature_group_paper_v2"
GROUP_SPLIT_VERSION = "hash_group_splitmix64_candidate_search_v2"
RUN_MODE = "paper"

DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/THAT_IDS_RERUN")
OUTPUT_ROOT = DRIVE_RUN_ROOT / EXPERIMENT_TAG / RUN_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reuse only dataset-level hashes/groups/audit from the successful pilot.
# Model checkpoints, scalers and split files are NOT reused from pilot.
REUSE_PILOT_DATA_CACHE = True
LEGACY_PILOT_ROOT = (
    DRIVE_RUN_ROOT
    / "conference_step4_exact_feature_group_v1"
    / "pilot"
)
NEW_SHARED_CACHE_ROOT = (
    DRIVE_RUN_ROOT
    / "conference_step4_shared_feature_cache_v1"
)
NEW_SHARED_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    "UNSW_V2": [
        "/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv",
        "/content/drive/MyDrive/NF-UNSW-NB15-v2.csv",
    ],
    "CIC_V2": [
        "/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv",
        "/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv",
    ],
}

# Có thể chạy cả hai liên tiếp. Nếu Colab thiếu RAM/thời gian, chạy UNSW trước,
# sau đó đổi thành ["CIC_V2"] và chạy lại; Cell 10 tự quét toàn bộ output đã có.
DATASETS_TO_RUN = ["UNSW_V2", "CIC_V2"]
PAPER_DATASETS = ["UNSW_V2", "CIC_V2"]

AUDIT_RANDOM_PROTOCOL = True
TRAIN_SPLITS = ["exact_feature_group"]
RUN_TRAINING = True

TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
TEST_FRAC = 0.20
TARGET_SPLIT_FRACTIONS = np.asarray(
    [TRAIN_FRAC, VAL_FRAC, TEST_FRAC], dtype=np.float64
)

HASH_CHUNK_SIZE = 250_000
GROUP_EVALUATION_CHUNK_SIZE = 1_000_000
MIN_FAMILY_TOTAL_FOR_BALANCE = 30
EXPECTED_FEATURE_COUNT = 39
ALLOW_FEATURE_COUNT_MISMATCH = False

WINDOW_SIZE = 5000
SCALER_CHUNK_SIZE = 500_000
CLIP_MIN = -10.0
CLIP_MAX = 10.0

HIDDEN_DIM = 96
AE_HIDDEN = 192
LATENT_DIM = 96
GRAPH_DROPOUT = 0.10
ENC_DROPOUT = 0.10
CLS_DROPOUT = 0.15
INIT_GATE_LOGIT = -1.5
CTX_RECON_WEIGHT = 0.10

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
SAVE_LAST_EVERY = 1

LAMBDA_RECON = 1.0
LAMBDA_CLS = 1.0
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
POS_WEIGHT_CAP = 12.0
SCORE_CLS_WEIGHT = 0.90
SCORE_RECON_WEIGHT = 0.10

EXCLUDE_COLS = [
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "Label",
    "Attack",
]

SEEDS = [42, 123, 456]
CORE_MODELS = [
    "mlp",
    "no_graph",
    "graph_classifier",
    "ae_only",
    "full",
]

# Recommended ON for the conference paper because it directly tests whether
# observed graph relations matter beyond model capacity.
RUN_SHUFFLED_EDGE_CONTROL = True
MODELS_TO_RUN = CORE_MODELS + (
    ["full_shuffled_edges"] if RUN_SHUFFLED_EDGE_CONTROL else []
)

MAX_EPOCHS = 80
PATIENCE = 12
GROUP_CANDIDATE_SALTS = 24
SMOKE_MAX_FLOWS_PER_SPLIT = None

# Aggregation never relies only on DATASETS_TO_RUN/MODELS_TO_RUN. It scans the
# existing paper output so running datasets in separate sessions cannot overwrite tables.
STRICT_FINAL_COMPLETENESS = False

print("Output:", OUTPUT_ROOT)
print("Legacy pilot cache:", LEGACY_PILOT_ROOT)
print("Datasets in this session:", DATASETS_TO_RUN)
print("Paper seeds:", SEEDS)
print("Models:", MODELS_TO_RUN)
print("Expected training tasks:", len(PAPER_DATASETS) * len(SEEDS) * len(MODELS_TO_RUN))


Output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_paper_v2/paper
Legacy pilot cache: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_v1/pilot
Datasets in this session: ['UNSW_V2', 'CIC_V2']
Paper seeds: [42, 123, 456]
Models: ['mlp', 'no_graph', 'graph_classifier', 'ae_only', 'full', 'full_shuffled_edges']
Expected training tasks: 36


In [3]:
# Cell 3 — Variants, cache resolution, and reproducibility helpers

@dataclass(frozen=True)
class VariantConfig:
    name: str
    graph_mode: str
    edge_control: str
    use_decoder: bool
    use_classifier: bool
    train_reconstruction: bool
    train_classifier: bool
    inference_mode: str


EXPERIMENT_VARIANTS: Dict[str, VariantConfig] = {
    "mlp": VariantConfig(
        name="mlp",
        graph_mode="none",
        edge_control="observed",
        use_decoder=False,
        use_classifier=True,
        train_reconstruction=False,
        train_classifier=True,
        inference_mode="classifier",
    ),
    "no_graph": VariantConfig(
        name="no_graph",
        graph_mode="none",
        edge_control="observed",
        use_decoder=True,
        use_classifier=True,
        train_reconstruction=True,
        train_classifier=True,
        inference_mode="hybrid",
    ),
    "graph_classifier": VariantConfig(
        name="graph_classifier",
        graph_mode="hetero",
        edge_control="observed",
        use_decoder=False,
        use_classifier=True,
        train_reconstruction=False,
        train_classifier=True,
        inference_mode="classifier",
    ),
    "ae_only": VariantConfig(
        name="ae_only",
        graph_mode="hetero",
        edge_control="observed",
        use_decoder=True,
        use_classifier=False,
        train_reconstruction=True,
        train_classifier=False,
        inference_mode="reconstruction",
    ),
    "full": VariantConfig(
        name="full",
        graph_mode="hetero",
        edge_control="observed",
        use_decoder=True,
        use_classifier=True,
        train_reconstruction=True,
        train_classifier=True,
        inference_mode="hybrid",
    ),
    "full_shuffled_edges": VariantConfig(
        name="full_shuffled_edges",
        graph_mode="hetero",
        edge_control="shuffled",
        use_decoder=True,
        use_classifier=True,
        train_reconstruction=True,
        train_classifier=True,
        inference_mode="hybrid",
    ),
}

unknown_models = sorted(set(MODELS_TO_RUN).difference(EXPERIMENT_VARIANTS))
if unknown_models:
    raise KeyError("Unknown MODELS_TO_RUN: " + str(unknown_models))


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state: Optional[Dict[str, Any]]) -> None:
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    # PyTorch >=2.6 defaults to weights_only=True; these checkpoints also contain
    # optimizer and RNG state, so explicitly request the complete object.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def resolve_dataset_path(dataset_name: str) -> str:
    candidates = DATA_PATH_CANDIDATES.get(dataset_name, [])
    for candidate in candidates:
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset " + dataset_name + ". Đã thử:\n" + "\n".join(candidates)
    )


CACHE_REQUIRED_FILES = [
    "exact_feature_hashes_uint64.npy",
    "feature_group_unique_hashes.npy",
    "feature_group_inverse_int32.npy",
    "feature_group_counts_int32.npy",
]


def resolve_dataset_cache_dir(dataset_name: str) -> Path:
    legacy = LEGACY_PILOT_ROOT / dataset_name
    if REUSE_PILOT_DATA_CACHE and all((legacy / name).exists() for name in CACHE_REQUIRED_FILES):
        print(f"Reuse pilot feature cache for {dataset_name}: {legacy}")
        return legacy
    cache_dir = NEW_SHARED_CACHE_ROOT / dataset_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    print(f"Use shared feature cache for {dataset_name}: {cache_dir}")
    return cache_dir


def split_configuration_fingerprint(
    dataset_name: str,
    protocol_name: str,
    seed: int,
    feature_names: List[str],
    n_rows: int,
    n_groups: int,
) -> str:
    payload = {
        "dataset": dataset_name,
        "protocol": protocol_name,
        "seed": int(seed),
        "split_version": GROUP_SPLIT_VERSION,
        "fractions": TARGET_SPLIT_FRACTIONS.tolist(),
        "candidate_salts": GROUP_CANDIDATE_SALTS if protocol_name == "exact_feature_group" else None,
        "features": feature_names,
        "n_rows": int(n_rows),
        "n_groups": int(n_groups),
        "run_mode": RUN_MODE,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:20]


def task_config_fingerprint(
    dataset_name: str,
    protocol_name: str,
    seed: int,
    feature_names: List[str],
    variant: VariantConfig,
    split_fingerprint: str,
) -> str:
    payload = {
        "experiment_tag": EXPERIMENT_TAG,
        "dataset": dataset_name,
        "protocol": protocol_name,
        "seed": int(seed),
        "variant": asdict(variant),
        "split_fingerprint": split_fingerprint,
        "features": feature_names,
        "window_size": WINDOW_SIZE,
        "hidden_dim": HIDDEN_DIM,
        "ae_hidden": AE_HIDDEN,
        "latent_dim": LATENT_DIM,
        "graph_dropout": GRAPH_DROPOUT,
        "encoder_dropout": ENC_DROPOUT,
        "classifier_dropout": CLS_DROPOUT,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "focal_alpha": FOCAL_ALPHA,
        "focal_gamma": FOCAL_GAMMA,
        "lambda_recon": LAMBDA_RECON,
        "lambda_cls": LAMBDA_CLS,
        "score_cls_weight": SCORE_CLS_WEIGHT,
        "score_recon_weight": SCORE_RECON_WEIGHT,
        "context_readout": "mean(src,dst,protocol)",
        "shuffled_control": "permute dst and protocol assignments within each graph batch",
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:20]


In [4]:
# Cell 4 — Dataset loading and scaler persistence

@dataclass
class DatasetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    attack_group_indices: List[np.ndarray]


def load_dataset_bundle(
    dataset_name: str,
    csv_path: str,
) -> DatasetBundle:
    path = Path(csv_path)
    if not path.exists():
        raise FileNotFoundError(path)

    print(f"\nLoading {dataset_name}: {path}")
    t0 = time.time()
    df = pd.read_csv(path, low_memory=False)

    required = {
        "IPV4_SRC_ADDR",
        "IPV4_DST_ADDR",
        "PROTOCOL",
        "Label",
        "Attack",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(
            "Dataset thiếu cột bắt buộc: "
            + str(sorted(missing))
        )

    feature_names = [
        c for c in df.columns
        if c not in EXCLUDE_COLS
    ]
    n_rows = len(df)
    n_features = len(feature_names)

    if (
        n_features != EXPECTED_FEATURE_COUNT
        and not ALLOW_FEATURE_COUNT_MISMATCH
    ):
        raise ValueError(
            f"Expected {EXPECTED_FEATURE_COUNT} model features "
            f"but found {n_features}. Features: {feature_names}"
        )

    print(
        f"Rows={n_rows:,} | "
        f"model-input features={n_features}"
    )

    # This is the exact numerical representation used by the model
    # before MinMax scaling.
    X_raw = np.empty(
        (n_rows, n_features),
        dtype=np.float32,
    )
    for j, col in enumerate(feature_names):
        arr = pd.to_numeric(
            df[col],
            errors="coerce",
        ).to_numpy(
            dtype=np.float32,
            copy=False,
        )
        X_raw[:, j] = np.nan_to_num(
            arr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        if (j + 1) % 10 == 0 or j + 1 == n_features:
            print(
                f"  Converted features: "
                f"{j + 1}/{n_features}"
            )

    src_ip = df["IPV4_SRC_ADDR"].to_numpy(copy=True)
    dst_ip = df["IPV4_DST_ADDR"].to_numpy(copy=True)
    protocol = df["PROTOCOL"].to_numpy(copy=True)
    labels = (
        pd.to_numeric(df["Label"], errors="coerce")
        .fillna(0)
        .to_numpy(dtype=np.int8)
    )

    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError("Label is not binary.")

    attack_codes, attack_classes = pd.factorize(
        df["Attack"].fillna("<MISSING_ATTACK>").astype(str),
        sort=False,
    )
    attack_codes = attack_codes.astype(
        np.int16,
        copy=False,
    )
    attack_classes = np.asarray(
        attack_classes,
        dtype=str,
    )

    order = np.argsort(
        attack_codes,
        kind="stable",
    )
    counts = np.bincount(
        attack_codes.astype(np.int64),
        minlength=len(attack_classes),
    )
    offsets = np.r_[0, np.cumsum(counts)]
    attack_group_indices = [
        order[offsets[i]:offsets[i + 1]].astype(
            np.int32,
            copy=True,
        )
        for i in range(len(attack_classes))
    ]

    del df
    gc.collect()

    print(
        f"Loaded in {(time.time() - t0) / 60:.1f} min"
    )
    print(
        "Label counts:",
        dict(zip(*np.unique(labels, return_counts=True))),
    )
    print("Attack classes:", len(attack_classes))

    return DatasetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=src_ip,
        dst_ip=dst_ip,
        protocol=protocol,
        labels=labels,
        attack_codes=attack_codes,
        attack_classes=attack_classes,
        attack_group_indices=attack_group_indices,
    )


def fit_or_load_scaler(
    bundle: DatasetBundle,
    train_idx: np.ndarray,
    shared_dir: Path,
    split_fingerprint: str,
) -> MinMaxScaler:
    scaler_path = shared_dir / "scaler.joblib"
    metadata_path = shared_dir / "scaler_metadata.json"

    if scaler_path.exists() and metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        if (
            metadata.get("split_fingerprint") == split_fingerprint
            and metadata.get("feature_names") == bundle.feature_names
            and metadata.get("train_rows") == int(len(train_idx))
        ):
            print("Loaded scaler:", scaler_path)
            return joblib.load(scaler_path)
        raise RuntimeError(
            f"Scaler cache mismatch at {shared_dir}. "
            "Use a new EXPERIMENT_TAG or remove this seed's shared folder."
        )

    if scaler_path.exists() or metadata_path.exists():
        raise RuntimeError(
            f"Incomplete scaler cache at {shared_dir}. Remove scaler.joblib and "
            "scaler_metadata.json, then rerun this seed."
        )

    scaler = MinMaxScaler()
    print("Fitting MinMaxScaler on exact-group-disjoint train only...")
    for start in range(0, len(train_idx), SCALER_CHUNK_SIZE):
        idx = train_idx[start:start + SCALER_CHUNK_SIZE]
        scaler.partial_fit(bundle.X_raw[idx])

    joblib.dump(scaler, scaler_path)
    atomic_json_dump(
        {
            "dataset": bundle.name,
            "split_fingerprint": split_fingerprint,
            "train_rows": int(len(train_idx)),
            "feature_names": bundle.feature_names,
            "fit_scope": "train_only",
        },
        metadata_path,
    )
    print("Saved scaler:", scaler_path)
    return scaler

def create_attack_stratified_random_split(
    bundle: DatasetBundle,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    train_parts = []
    val_parts = []
    test_parts = []

    for group_idx in bundle.attack_group_indices:
        perm = rng.permutation(group_idx)
        n = len(perm)
        cut_train = int(n * TRAIN_FRAC)
        cut_val = int(n * (TRAIN_FRAC + VAL_FRAC))
        train_parts.append(perm[:cut_train])
        val_parts.append(perm[cut_train:cut_val])
        test_parts.append(perm[cut_val:])

    train_idx = rng.permutation(
        np.concatenate(train_parts)
    ).astype(np.int32, copy=False)
    val_idx = rng.permutation(
        np.concatenate(val_parts)
    ).astype(np.int32, copy=False)
    test_idx = rng.permutation(
        np.concatenate(test_parts)
    ).astype(np.int32, copy=False)

    return train_idx, val_idx, test_idx


def smoke_limit_indices(
    idx: np.ndarray,
    bundle: DatasetBundle,
    seed: int,
) -> np.ndarray:
    if (
        SMOKE_MAX_FLOWS_PER_SPLIT is None
        or len(idx) <= SMOKE_MAX_FLOWS_PER_SPLIT
    ):
        return idx.astype(np.int32, copy=False)

    rng = np.random.default_rng(seed)
    y = bundle.labels[idx]
    positives = idx[y == 1]
    negatives = idx[y == 0]

    positive_fraction = len(positives) / len(idx)
    n_positive = min(
        len(positives),
        max(
            1,
            int(
                round(
                    SMOKE_MAX_FLOWS_PER_SPLIT
                    * positive_fraction
                )
            ),
        ),
    )
    n_negative = min(
        len(negatives),
        SMOKE_MAX_FLOWS_PER_SPLIT - n_positive,
    )

    selected = np.concatenate([
        rng.choice(
            positives,
            size=n_positive,
            replace=False,
        ),
        rng.choice(
            negatives,
            size=n_negative,
            replace=False,
        ),
    ])
    return rng.permutation(selected).astype(
        np.int32,
        copy=False,
    )


In [5]:
# Cell 5 — Exact feature hashes, duplicate audit, and group-disjoint split

def compute_or_load_feature_hashes(
    bundle: DatasetBundle,
    dataset_dir: Path,
) -> np.ndarray:
    hash_path = (
        dataset_dir
        / "exact_feature_hashes_uint64.npy"
    )
    if hash_path.exists():
        hashes = np.load(hash_path, mmap_mode="r")
        if len(hashes) == len(bundle.labels):
            print("Loaded feature hashes:", hash_path)
            return hashes
        hash_path.unlink()

    print(
        "Computing uint64 hashes of exact float32 "
        "model-input vectors before scaling..."
    )
    hashes = np.empty(
        len(bundle.labels),
        dtype=np.uint64,
    )

    for start in range(
        0,
        len(bundle.labels),
        HASH_CHUNK_SIZE,
    ):
        end = min(
            start + HASH_CHUNK_SIZE,
            len(bundle.labels),
        )
        frame = pd.DataFrame(
            bundle.X_raw[start:end],
            columns=bundle.feature_names,
            copy=False,
        )
        hashes[start:end] = (
            pd.util.hash_pandas_object(
                frame,
                index=False,
            )
            .to_numpy(dtype=np.uint64)
        )
        if (
            end == len(bundle.labels)
            or end % 1_000_000 == 0
        ):
            print(
                f"  Hashed {end:,}/"
                f"{len(bundle.labels):,}"
            )
        del frame

    np.save(hash_path, hashes)
    print("Saved feature hashes:", hash_path)
    return np.load(hash_path, mmap_mode="r")


def compute_or_load_feature_groups(
    hashes: np.ndarray,
    dataset_dir: Path,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    unique_path = dataset_dir / "feature_group_unique_hashes.npy"
    inverse_path = dataset_dir / "feature_group_inverse_int32.npy"
    counts_path = dataset_dir / "feature_group_counts_int32.npy"

    if (
        unique_path.exists()
        and inverse_path.exists()
        and counts_path.exists()
    ):
        unique_hashes = np.load(unique_path, mmap_mode="r")
        inverse = np.load(inverse_path, mmap_mode="r")
        counts = np.load(counts_path, mmap_mode="r")
        if len(inverse) == len(hashes):
            print(
                "Loaded feature-group cache:",
                f"{len(unique_hashes):,} groups",
            )
            return unique_hashes, inverse, counts

    print("Building exact-feature group IDs...")
    t0 = time.time()
    unique_hashes, inverse, counts = np.unique(
        np.asarray(hashes),
        return_inverse=True,
        return_counts=True,
    )

    if len(unique_hashes) >= np.iinfo(np.int32).max:
        raise OverflowError(
            "Too many feature groups for int32 inverse."
        )

    inverse = inverse.astype(np.int32, copy=False)
    counts = counts.astype(np.int32, copy=False)

    np.save(unique_path, unique_hashes)
    np.save(inverse_path, inverse)
    np.save(counts_path, counts)

    print(
        f"Saved {len(unique_hashes):,} groups in "
        f"{(time.time() - t0) / 60:.1f} min"
    )
    del unique_hashes, inverse, counts
    gc.collect()

    return (
        np.load(unique_path, mmap_mode="r"),
        np.load(inverse_path, mmap_mode="r"),
        np.load(counts_path, mmap_mode="r"),
    )


def run_duplicate_audit(
    bundle: DatasetBundle,
    hashes: np.ndarray,
    group_inverse: np.ndarray,
    group_counts: np.ndarray,
    dataset_dir: Path,
) -> Dict[str, Any]:
    summary_path = dataset_dir / "duplicate_audit_summary.json"
    attack_path = dataset_dir / "duplicate_audit_per_attack.csv"
    feature_path = dataset_dir / "model_feature_names.csv"

    pd.DataFrame({
        "feature_index": np.arange(len(bundle.feature_names)),
        "feature": bundle.feature_names,
    }).to_csv(feature_path, index=False)

    if summary_path.exists() and attack_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if (
            saved.get("n_rows") == int(len(bundle.labels))
            and saved.get("n_features") == int(len(bundle.feature_names))
        ):
            print("Loaded duplicate audit:", summary_path)
            return saved
        print("Cached duplicate audit does not match this dataset; recomputing.")

    print("Running duplicate and conflict audit...")
    n_groups = len(group_counts)
    labels = bundle.labels
    attacks = bundle.attack_codes
    inverse = np.asarray(group_inverse)

    label_min = np.full(
        n_groups,
        2,
        dtype=np.int8,
    )
    label_max = np.full(
        n_groups,
        -1,
        dtype=np.int8,
    )
    np.minimum.at(label_min, inverse, labels)
    np.maximum.at(label_max, inverse, labels)

    attack_min = np.full(
        n_groups,
        np.iinfo(np.int16).max,
        dtype=np.int16,
    )
    attack_max = np.full(
        n_groups,
        -1,
        dtype=np.int16,
    )
    np.minimum.at(attack_min, inverse, attacks)
    np.maximum.at(attack_max, inverse, attacks)

    counts = np.asarray(group_counts, dtype=np.int64)
    duplicate_mask = counts > 1
    label_conflict = label_min != label_max
    attack_conflict = attack_min != attack_max

    repeated_rows = int(
        len(labels) - n_groups
    )
    rows_in_duplicate_groups = int(
        counts[duplicate_mask].sum()
    )

    summary = {
        "dataset": bundle.name,
        "n_rows": int(len(labels)),
        "n_features": int(len(bundle.feature_names)),
        "n_unique_feature_groups": int(n_groups),
        "n_duplicate_groups": int(duplicate_mask.sum()),
        "repeated_rows_beyond_first": repeated_rows,
        "repeated_rows_rate": float(
            repeated_rows / max(len(labels), 1)
        ),
        "rows_belonging_to_duplicate_groups": (
            rows_in_duplicate_groups
        ),
        "rows_in_duplicate_groups_rate": float(
            rows_in_duplicate_groups
            / max(len(labels), 1)
        ),
        "maximum_group_size": int(counts.max()),
        "group_size_q50": float(np.quantile(counts, 0.50)),
        "group_size_q90": float(np.quantile(counts, 0.90)),
        "group_size_q99": float(np.quantile(counts, 0.99)),
        "binary_label_conflicting_groups": int(
            label_conflict.sum()
        ),
        "rows_in_binary_label_conflicting_groups": int(
            counts[label_conflict].sum()
        ),
        "attack_name_conflicting_groups": int(
            attack_conflict.sum()
        ),
        "rows_in_attack_name_conflicting_groups": int(
            counts[attack_conflict].sum()
        ),
        "hash_definition": (
            "pandas uint64 hash over the float32 numerical "
            "model-input vector after NaN/Inf normalization "
            "and before MinMaxScaler"
        ),
        "hash_collision_note": (
            "A deterministic 64-bit hash is used. Collision "
            "risk is negligible at this scale but nonzero."
        ),
    }

    per_attack = []
    raw_hashes = np.asarray(hashes)
    for code_value, attack_name in enumerate(
        bundle.attack_classes
    ):
        mask = attacks == code_value
        count = int(mask.sum())
        if count == 0:
            continue
        unique_count = int(
            np.unique(raw_hashes[mask]).size
        )
        repeated = count - unique_count
        per_attack.append({
            "attack": str(attack_name),
            "count": count,
            "unique_feature_vectors": unique_count,
            "repeated_rows_beyond_first": repeated,
            "repeated_rate": float(repeated / count),
        })

    atomic_json_dump(summary, summary_path)
    pd.DataFrame(per_attack).sort_values(
        ["repeated_rate", "count"],
        ascending=[False, False],
    ).to_csv(attack_path, index=False)

    print(json.dumps(summary, indent=2, ensure_ascii=False))

    del (
        label_min,
        label_max,
        attack_min,
        attack_max,
    )
    gc.collect()
    return summary


def splitmix64(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.uint64).copy()
    with np.errstate(over="ignore"):
        x = x + np.uint64(0x9E3779B97F4A7C15)
        x = (
            (x ^ (x >> np.uint64(30)))
            * np.uint64(0xBF58476D1CE4E5B9)
        )
        x = (
            (x ^ (x >> np.uint64(27)))
            * np.uint64(0x94D049BB133111EB)
        )
        x = x ^ (x >> np.uint64(31))
    return x


def evaluate_group_assignment(
    bundle: DatasetBundle,
    group_inverse: np.ndarray,
    group_counts: np.ndarray,
    group_assignment: np.ndarray,
) -> Dict[str, Any]:
    counts = np.asarray(group_counts, dtype=np.int64)
    split_rows = np.bincount(
        group_assignment.astype(np.int64),
        weights=counts,
        minlength=3,
    ).astype(np.int64)

    n_attacks = len(bundle.attack_classes)
    split_attack = np.zeros(
        (3, n_attacks),
        dtype=np.int64,
    )

    for start in range(
        0,
        len(bundle.labels),
        GROUP_EVALUATION_CHUNK_SIZE,
    ):
        end = min(
            start + GROUP_EVALUATION_CHUNK_SIZE,
            len(bundle.labels),
        )
        split_code = group_assignment[
            np.asarray(group_inverse[start:end])
        ].astype(np.int64)
        combined = (
            split_code * n_attacks
            + bundle.attack_codes[start:end].astype(np.int64)
        )
        chunk_counts = np.bincount(
            combined,
            minlength=3 * n_attacks,
        ).reshape(3, n_attacks)
        split_attack += chunk_counts

    row_fractions = (
        split_rows / max(split_rows.sum(), 1)
    )
    global_error = float(
        np.abs(
            row_fractions
            - TARGET_SPLIT_FRACTIONS
        ).mean()
    )

    totals = split_attack.sum(axis=0)
    eligible = totals >= MIN_FAMILY_TOTAL_FOR_BALANCE
    family_fraction = np.divide(
        split_attack,
        totals[None, :],
        out=np.zeros_like(
            split_attack,
            dtype=np.float64,
        ),
        where=totals[None, :] > 0,
    )
    family_error_matrix = np.abs(
        family_fraction
        - TARGET_SPLIT_FRACTIONS[:, None]
    )

    if eligible.any():
        weights = np.sqrt(totals[eligible].astype(np.float64))
        weights /= weights.sum()
        family_mae = float(
            (
                family_error_matrix[:, eligible].mean(axis=0)
                * weights
            ).sum()
        )
        max_family_error = float(
            family_error_matrix[:, eligible].max()
        )
    else:
        family_mae = 0.0
        max_family_error = 0.0

    expected = (
        TARGET_SPLIT_FRACTIONS[:, None]
        * totals[None, :]
    )
    missing_required = int(
        (
            (split_attack == 0)
            & (expected >= 5)
        ).sum()
    )

    objective = (
        5.0 * global_error
        + family_mae
        + 0.25 * max_family_error
        + 10.0 * missing_required
    )

    return {
        "objective": float(objective),
        "global_fraction_mae": global_error,
        "family_fraction_weighted_mae": family_mae,
        "max_family_fraction_error": max_family_error,
        "missing_required_family_cells": missing_required,
        "split_rows": split_rows.tolist(),
        "split_row_fractions": row_fractions.tolist(),
        "split_attack_counts": split_attack,
    }


def create_exact_feature_group_split(
    bundle: DatasetBundle,
    unique_hashes: np.ndarray,
    group_inverse: np.ndarray,
    group_counts: np.ndarray,
    seed: int,
    shared_dir: Path,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    rng = np.random.default_rng(seed)
    salts = rng.integers(
        1,
        np.iinfo(np.uint64).max,
        size=GROUP_CANDIDATE_SALTS,
        dtype=np.uint64,
    )

    best = None
    best_assignment = None
    candidate_rows = []

    print(
        f"Searching {GROUP_CANDIDATE_SALTS} deterministic "
        "group assignments..."
    )

    unique_values = np.asarray(unique_hashes)
    for candidate_index, salt in enumerate(salts):
        mixed = splitmix64(
            unique_values ^ salt
        )
        unit = mixed.astype(np.float64) / float(2**64)
        assignment = np.where(
            unit < TRAIN_FRAC,
            0,
            np.where(
                unit < TRAIN_FRAC + VAL_FRAC,
                1,
                2,
            ),
        ).astype(np.int8)

        metrics = evaluate_group_assignment(
            bundle,
            group_inverse,
            group_counts,
            assignment,
        )
        candidate_rows.append({
            "candidate": candidate_index,
            "salt": int(salt),
            **{
                key: value
                for key, value in metrics.items()
                if key != "split_attack_counts"
            },
        })

        print(
            f"  candidate={candidate_index:02d} | "
            f"objective={metrics['objective']:.6f} | "
            f"rows={metrics['split_row_fractions']} | "
            f"missing={metrics['missing_required_family_cells']}"
        )

        if (
            best is None
            or metrics["objective"] < best["objective"]
        ):
            best = metrics
            best_assignment = assignment.copy()
            best_salt = int(salt)

    if best_assignment is None:
        raise RuntimeError("No group assignment was created.")

    pd.DataFrame(candidate_rows).to_csv(
        shared_dir / "group_assignment_candidates.csv",
        index=False,
    )
    np.save(
        shared_dir / "feature_group_assignment_int8.npy",
        best_assignment,
    )

    row_assignment = best_assignment[
        np.asarray(group_inverse)
    ]
    train_idx = np.flatnonzero(
        row_assignment == 0
    )
    val_idx = np.flatnonzero(
        row_assignment == 1
    )
    test_idx = np.flatnonzero(
        row_assignment == 2
    )

    shuffle_rng = np.random.default_rng(seed + 17_171)
    train_idx = shuffle_rng.permutation(train_idx)
    val_idx = shuffle_rng.permutation(val_idx)
    test_idx = shuffle_rng.permutation(test_idx)

    train_idx = smoke_limit_indices(
        train_idx,
        bundle,
        seed + 101,
    )
    val_idx = smoke_limit_indices(
        val_idx,
        bundle,
        seed + 103,
    )
    test_idx = smoke_limit_indices(
        test_idx,
        bundle,
        seed + 107,
    )

    metadata = {
        "protocol": (
            "exact-feature-group-disjoint 70/10/20 split; "
            "all rows sharing one pre-scaler numerical "
            "model-input vector remain in one subset"
        ),
        "split_algorithm_version": GROUP_SPLIT_VERSION,
        "n_feature_groups": int(len(unique_hashes)),
        "candidate_salts_tested": int(
            GROUP_CANDIDATE_SALTS
        ),
        "selected_salt": best_salt,
        "selected_objective": float(best["objective"]),
        "selected_global_fraction_mae": float(
            best["global_fraction_mae"]
        ),
        "selected_family_fraction_weighted_mae": float(
            best["family_fraction_weighted_mae"]
        ),
        "selected_max_family_fraction_error": float(
            best["max_family_fraction_error"]
        ),
        "selected_missing_required_family_cells": int(
            best["missing_required_family_cells"]
        ),
        "full_assignment_split_rows": best["split_rows"],
        "full_assignment_split_row_fractions": (
            best["split_row_fractions"]
        ),
    }

    del row_assignment, best_assignment
    gc.collect()
    return (
        train_idx.astype(np.int32, copy=False),
        val_idx.astype(np.int32, copy=False),
        test_idx.astype(np.int32, copy=False),
        metadata,
    )


def split_label_stats(
    bundle: DatasetBundle,
    idx: np.ndarray,
) -> Dict[str, Any]:
    y = bundle.labels[idx]
    return {
        "rows": int(len(idx)),
        "benign": int((y == 0).sum()),
        "attack": int((y == 1).sum()),
        "attack_rate": float(y.mean()) if len(y) else 0.0,
    }


def validate_row_disjoint(
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
) -> None:
    if (
        len(np.intersect1d(train_idx, val_idx))
        or len(np.intersect1d(train_idx, test_idx))
        or len(np.intersect1d(val_idx, test_idx))
    ):
        raise RuntimeError("Row indices overlap across splits.")


def split_overlap_audit(
    bundle: DatasetBundle,
    group_inverse: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
) -> Dict[str, Any]:
    n_groups = int(
        np.max(group_inverse)
    ) + 1
    group_presence = np.zeros(
        n_groups,
        dtype=np.uint8,
    )

    train_groups = np.asarray(group_inverse[train_idx])
    val_groups = np.asarray(group_inverse[val_idx])
    test_groups = np.asarray(group_inverse[test_idx])

    np.bitwise_or.at(
        group_presence,
        train_groups,
        np.uint8(1),
    )
    np.bitwise_or.at(
        group_presence,
        val_groups,
        np.uint8(2),
    )
    np.bitwise_or.at(
        group_presence,
        test_groups,
        np.uint8(4),
    )

    train_present = (group_presence & 1) != 0
    val_seen = train_present[val_groups]
    test_seen = train_present[test_groups]

    train_label_bits = np.zeros(
        n_groups,
        dtype=np.uint8,
    )
    train_label_value_bits = np.where(
        bundle.labels[train_idx] == 0,
        np.uint8(1),
        np.uint8(2),
    )
    np.bitwise_or.at(
        train_label_bits,
        train_groups,
        train_label_value_bits,
    )

    val_required_bits = np.where(
        bundle.labels[val_idx] == 0,
        np.uint8(1),
        np.uint8(2),
    )
    test_required_bits = np.where(
        bundle.labels[test_idx] == 0,
        np.uint8(1),
        np.uint8(2),
    )
    val_same_label_seen = (
        train_label_bits[val_groups]
        & val_required_bits
    ) != 0
    test_same_label_seen = (
        train_label_bits[test_groups]
        & test_required_bits
    ) != 0

    return {
        "train_rows": int(len(train_idx)),
        "validation_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "validation_rows_with_feature_group_in_train": int(
            val_seen.sum()
        ),
        "validation_feature_overlap_rate": float(
            val_seen.mean()
        ) if len(val_seen) else 0.0,
        "test_rows_with_feature_group_in_train": int(
            test_seen.sum()
        ),
        "test_feature_overlap_rate": float(
            test_seen.mean()
        ) if len(test_seen) else 0.0,
        "validation_same_feature_and_label_in_train": int(
            val_same_label_seen.sum()
        ),
        "validation_same_feature_and_label_overlap_rate": float(
            val_same_label_seen.mean()
        ) if len(val_same_label_seen) else 0.0,
        "test_same_feature_and_label_in_train": int(
            test_same_label_seen.sum()
        ),
        "test_same_feature_and_label_overlap_rate": float(
            test_same_label_seen.mean()
        ) if len(test_same_label_seen) else 0.0,
        "unique_groups_shared_train_validation": int(
            np.sum((group_presence & 3) == 3)
        ),
        "unique_groups_shared_train_test": int(
            np.sum((group_presence & 5) == 5)
        ),
        "unique_groups_shared_validation_test": int(
            np.sum((group_presence & 6) == 6)
        ),
        "unique_groups_spanning_all_three": int(
            np.sum(group_presence == 7)
        ),
    }


def save_split_attack_distribution(
    bundle: DatasetBundle,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    path: Path,
) -> None:
    rows = []
    total_by_attack = np.bincount(
        bundle.attack_codes.astype(np.int64),
        minlength=len(bundle.attack_classes),
    )
    for split_code, (split_name, idx) in enumerate([
        ("train", train_idx),
        ("validation", val_idx),
        ("test", test_idx),
    ]):
        counts = np.bincount(
            bundle.attack_codes[idx].astype(np.int64),
            minlength=len(bundle.attack_classes),
        )
        target_fraction = float(TARGET_SPLIT_FRACTIONS[split_code])
        for code_value, attack_name in enumerate(bundle.attack_classes):
            count = int(counts[code_value])
            family_total = int(total_by_attack[code_value])
            fraction_of_family = float(count / family_total) if family_total else 0.0
            rows.append({
                "split": split_name,
                "attack": str(attack_name),
                "count": count,
                "split_rows": int(len(idx)),
                "fraction_within_split": float(count / max(len(idx), 1)),
                "family_total": family_total,
                "fraction_of_attack_family": fraction_of_family,
                "target_fraction": target_fraction,
                "absolute_fraction_error": abs(fraction_of_family - target_fraction),
            })
    pd.DataFrame(rows).to_csv(path, index=False)

def get_or_create_protocol_split(
    bundle: DatasetBundle,
    protocol_name: str,
    seed: int,
    unique_hashes: np.ndarray,
    group_inverse: np.ndarray,
    group_counts: np.ndarray,
    shared_dir: Path,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    split_path = shared_dir / "split_indices.npz"
    metadata_path = shared_dir / "split_metadata.json"
    shared_dir.mkdir(parents=True, exist_ok=True)
    expected_split_fingerprint = split_configuration_fingerprint(
        dataset_name=bundle.name,
        protocol_name=protocol_name,
        seed=seed,
        feature_names=bundle.feature_names,
        n_rows=len(bundle.labels),
        n_groups=len(unique_hashes),
    )

    if split_path.exists() and metadata_path.exists():
        data = np.load(split_path)
        train_idx = data["train_idx"]
        val_idx = data["val_idx"]
        test_idx = data["test_idx"]
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        if metadata.get("protocol_name") != protocol_name:
            raise RuntimeError("Cached protocol mismatch: " + str(shared_dir))
        if metadata.get("split_fingerprint") != expected_split_fingerprint:
            raise RuntimeError(
                f"Cached split fingerprint mismatch at {shared_dir}. "
                "Use a new EXPERIMENT_TAG or remove this seed's shared folder."
            )
        validate_row_disjoint(
            train_idx,
            val_idx,
            test_idx,
        )
        print("Loaded split:", split_path)
        return train_idx, val_idx, test_idx, metadata

    if protocol_name == "random":
        train_idx, val_idx, test_idx = (
            create_attack_stratified_random_split(
                bundle,
                seed,
            )
        )
        train_idx = smoke_limit_indices(
            train_idx,
            bundle,
            seed + 201,
        )
        val_idx = smoke_limit_indices(
            val_idx,
            bundle,
            seed + 203,
        )
        test_idx = smoke_limit_indices(
            test_idx,
            bundle,
            seed + 207,
        )
        protocol_metadata = {
            "protocol": (
                "attack-category-stratified random split "
                "followed by independent split shuffling"
            ),
        }
    elif protocol_name == "exact_feature_group":
        (
            train_idx,
            val_idx,
            test_idx,
            protocol_metadata,
        ) = create_exact_feature_group_split(
            bundle,
            unique_hashes,
            group_inverse,
            group_counts,
            seed,
            shared_dir,
        )
    else:
        raise ValueError(
            "Unknown protocol: " + protocol_name
        )

    validate_row_disjoint(
        train_idx,
        val_idx,
        test_idx,
    )

    for split_name, idx in [
        ("train", train_idx),
        ("validation", val_idx),
        ("test", test_idx),
    ]:
        if np.unique(bundle.labels[idx]).size < 2:
            raise RuntimeError(
                split_name
                + " does not contain both binary classes."
            )

    overlap = split_overlap_audit(
        bundle,
        group_inverse,
        train_idx,
        val_idx,
        test_idx,
    )

    if protocol_name == "exact_feature_group":
        if (
            overlap["validation_feature_overlap_rate"] != 0.0
            or overlap["test_feature_overlap_rate"] != 0.0
            or overlap["unique_groups_spanning_all_three"] != 0
        ):
            raise RuntimeError(
                "Exact-feature-group split failed overlap audit."
            )

    np.savez_compressed(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

    metadata = {
        "dataset": bundle.name,
        "protocol_name": protocol_name,
        "seed": int(seed),
        "run_mode": RUN_MODE,
        "split_fingerprint": expected_split_fingerprint,
        "feature_names": bundle.feature_names,
        "train": split_label_stats(bundle, train_idx),
        "validation": split_label_stats(bundle, val_idx),
        "test": split_label_stats(bundle, test_idx),
        "overlap_audit": overlap,
        **protocol_metadata,
    }
    atomic_json_dump(metadata, metadata_path)
    atomic_json_dump(
        overlap,
        shared_dir / "split_overlap_audit.json",
    )
    save_split_attack_distribution(
        bundle,
        train_idx,
        val_idx,
        test_idx,
        shared_dir / "split_attack_distribution.csv",
    )

    print(
        protocol_name,
        json.dumps(overlap, indent=2),
    )
    return train_idx, val_idx, test_idx, metadata


In [6]:

# Cell 6 — Conference-compatible graph batches and model definitions

@dataclass
class GraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    src: torch.Tensor
    dst: torch.Tensor
    proto_local: torch.Tensor
    shuffled_dst: torch.Tensor
    shuffled_proto_local: torch.Tensor
    n_ip: int
    n_proto: int


def build_graph_window(
    bundle: DatasetBundle,
    row_idx: np.ndarray,
    scaler: MinMaxScaler,
    control_seed: int,
) -> GraphWindow:
    # Fixed-size non-temporal graph batch in the shuffled split order.
    raw_feat = bundle.X_raw[row_idx]
    scaled_feat = scaler.transform(raw_feat)
    scaled_feat = np.clip(scaled_feat, CLIP_MIN, CLIP_MAX).astype(np.float32, copy=False)

    src_values = bundle.src_ip[row_idx]
    dst_values = bundle.dst_ip[row_idx]
    proto_values = bundle.protocol[row_idx]

    # Normalize missing entity identifiers before local factorization.
    src_values = pd.Series(src_values, copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    dst_values = pd.Series(dst_values, copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    proto_values = pd.Series(proto_values, copy=False).fillna("<MISSING_PROTOCOL>").astype(str).to_numpy()

    # Window-local node identifiers. The same IP receives the same ID whether it
    # appears as source or destination inside this graph batch.
    combined_ip = np.concatenate([src_values, dst_values])
    ip_codes, _ = pd.factorize(combined_ip, sort=False)
    n = len(row_idx)
    src = ip_codes[:n].astype(np.int64, copy=False)
    dst = ip_codes[n:].astype(np.int64, copy=False)
    proto_local, proto_uniques = pd.factorize(proto_values, sort=False)
    proto_local = proto_local.astype(np.int64, copy=False)

    # Randomized-topology control: preserve the exact multiset/frequency of
    # destination and protocol assignments, but break their association with flows.
    control_rng = np.random.default_rng(control_seed)
    shuffled_dst = dst[control_rng.permutation(n)].astype(np.int64, copy=False)
    shuffled_proto_local = proto_local[control_rng.permutation(n)].astype(np.int64, copy=False)

    return GraphWindow(
        edge_feat=torch.from_numpy(scaled_feat),
        labels=torch.from_numpy(bundle.labels[row_idx].astype(np.int64, copy=False)),
        src=torch.from_numpy(src),
        dst=torch.from_numpy(dst),
        proto_local=torch.from_numpy(proto_local),
        shuffled_dst=torch.from_numpy(shuffled_dst),
        shuffled_proto_local=torch.from_numpy(shuffled_proto_local),
        n_ip=int(ip_codes.max() + 1) if len(ip_codes) else 0,
        n_proto=int(len(proto_uniques)),
    )


def prepare_graph_windows(
    bundle: DatasetBundle,
    split_idx: np.ndarray,
    scaler: MinMaxScaler,
    description: str,
    control_seed: int,
) -> List[GraphWindow]:
    windows: List[GraphWindow] = []
    n_windows = math.ceil(len(split_idx) / WINDOW_SIZE)
    t0 = time.time()
    for w in range(n_windows):
        idx = split_idx[w * WINDOW_SIZE:(w + 1) * WINDOW_SIZE]
        if len(idx) == 0:
            continue
        windows.append(
            build_graph_window(
                bundle,
                idx,
                scaler,
                control_seed=control_seed + 1_000_003 * w,
            )
        )
        if (w + 1) % 500 == 0 or w + 1 == n_windows:
            print(f"    {description}: {w + 1}/{n_windows} graph batches")
    print(f"  {description}: {len(windows)} batches built in {(time.time() - t0) / 60:.1f} min")
    return windows


class RelationAttentionEncoder(nn.Module):
    """Relation-specific edge attention followed by semantic attention."""

    def __init__(self, in_dim: int, out_dim: int, n_relations: int, dropout: float = 0.10):
        super().__init__()
        if n_relations < 1:
            raise ValueError("n_relations must be >= 1")
        self.out_dim = out_dim
        self.n_relations = n_relations
        self.proj = nn.Linear(in_dim, out_dim)
        self.edge_attention = nn.ModuleList([
            nn.Linear(out_dim * 2, 1) for _ in range(n_relations)
        ])
        self.semantic_attention = nn.Linear(out_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.norms = nn.ModuleList([nn.LayerNorm(out_dim) for _ in range(n_relations)])

    def aggregate_relation(
        self,
        h: torch.Tensor,
        edge_src: torch.Tensor,
        edge_dst: torch.Tensor,
        relation_idx: int,
    ) -> torch.Tensor:
        if edge_src.numel() == 0:
            return h

        src_h = h[edge_src]
        dst_h = h[edge_dst]
        alpha = torch.sigmoid(
            self.edge_attention[relation_idx](torch.cat([src_h, dst_h], dim=1))
        )
        messages = src_h * alpha

        out = torch.zeros_like(h)
        weight_sum = torch.zeros(h.size(0), 1, device=h.device, dtype=h.dtype)
        out.scatter_add_(0, edge_dst.unsqueeze(1).expand(-1, self.out_dim), messages)
        weight_sum.scatter_add_(0, edge_dst.unsqueeze(1), alpha)
        out = out / weight_sum.clamp(min=1.0)
        return self.norms[relation_idx](h + self.dropout(out))

    def forward(
        self,
        node_features: torch.Tensor,
        relations: List[Tuple[torch.Tensor, torch.Tensor]],
    ) -> torch.Tensor:
        if len(relations) != self.n_relations:
            raise ValueError(f"Expected {self.n_relations} relations, received {len(relations)}")

        h = F.relu(self.proj(node_features))
        relation_outputs = [
            self.aggregate_relation(h, src, dst, relation_idx)
            for relation_idx, (src, dst) in enumerate(relations)
        ]

        if len(relation_outputs) == 1:
            return relation_outputs[0]

        stacked = torch.stack(relation_outputs, dim=0)  # [R, N, H]
        semantic_logits = self.semantic_attention(stacked)
        semantic_weights = torch.softmax(semantic_logits, dim=0)
        return (semantic_weights * stacked).sum(dim=0)


class HybridDetector(nn.Module):
    def __init__(
        self,
        input_dim: int,
        flow_dim: int,
        hidden_dim: int,
        latent_dim: int,
        use_decoder: bool,
        use_classifier: bool,
        ctx_recon_weight: float,
    ):
        super().__init__()
        self.flow_dim = flow_dim
        self.ctx_recon_weight = ctx_recon_weight
        self.use_decoder = use_decoder
        self.use_classifier = use_classifier

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(ENC_DROPOUT),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )

        self.decoder = None
        if use_decoder:
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
            )

        self.classifier = None
        if use_classifier:
            self.classifier = nn.Sequential(
                nn.Linear(input_dim + latent_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(CLS_DROPOUT),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1),
            )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:
        z = self.encoder(x)
        reconstruction = self.decoder(z) if self.decoder is not None else None
        logits = self.classifier(torch.cat([x, z], dim=1)).squeeze(1) if self.classifier is not None else None
        return z, reconstruction, logits

    def reconstruction_error(self, x: torch.Tensor, reconstruction: torch.Tensor) -> torch.Tensor:
        err = (reconstruction - x) ** 2
        if self.flow_dim < x.size(1):
            flow_err = err[:, :self.flow_dim].mean(dim=1)
            context_err = err[:, self.flow_dim:].mean(dim=1)
            return flow_err + self.ctx_recon_weight * context_err
        return err.mean(dim=1)


class THATIDSVariantModel(nn.Module):
    def __init__(self, flow_dim: int, variant: VariantConfig):
        super().__init__()
        self.flow_dim = flow_dim
        self.variant = variant

        if variant.graph_mode == "hetero":
            self.graph_encoder = RelationAttentionEncoder(
                in_dim=flow_dim,
                out_dim=HIDDEN_DIM,
                n_relations=2,
                dropout=GRAPH_DROPOUT,
            )
            self.context_gate = nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
            detector_input_dim = flow_dim + HIDDEN_DIM
        elif variant.graph_mode == "homo":
            self.graph_encoder = RelationAttentionEncoder(
                in_dim=flow_dim,
                out_dim=HIDDEN_DIM,
                n_relations=1,
                dropout=GRAPH_DROPOUT,
            )
            self.context_gate = nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
            detector_input_dim = flow_dim + HIDDEN_DIM
        elif variant.graph_mode == "none":
            self.graph_encoder = None
            self.register_parameter("context_gate", None)
            detector_input_dim = flow_dim
        else:
            raise ValueError(f"Unknown graph mode: {variant.graph_mode}")

        self.detector = HybridDetector(
            input_dim=detector_input_dim,
            flow_dim=flow_dim,
            hidden_dim=AE_HIDDEN,
            latent_dim=LATENT_DIM,
            use_decoder=variant.use_decoder,
            use_classifier=variant.use_classifier,
            ctx_recon_weight=CTX_RECON_WEIGHT,
        )

    def gate_value(self) -> Optional[float]:
        if self.context_gate is None:
            return None
        return float(torch.sigmoid(self.context_gate.detach()).item())

    def prepare_flow_input(self, window: GraphWindow, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        flow_features = window.edge_feat.to(device, non_blocking=True)
        labels = window.labels.to(device, non_blocking=True)

        if self.variant.graph_mode == "none":
            return flow_features, labels

        src = window.src.to(device, non_blocking=True)
        if self.variant.edge_control == "shuffled":
            dst = window.shuffled_dst.to(device, non_blocking=True)
            proto_local = window.shuffled_proto_local.to(device, non_blocking=True)
        elif self.variant.edge_control == "observed":
            dst = window.dst.to(device, non_blocking=True)
            proto_local = window.proto_local.to(device, non_blocking=True)
        else:
            raise ValueError(f"Unknown edge control: {self.variant.edge_control}")
        proto = proto_local + window.n_ip
        n_nodes = window.n_ip + window.n_proto
        n_flows = flow_features.size(0)

        # Initial node feature = mean of incident flow vectors for source, destination,
        # and protocol roles inside this graph batch.
        node_features = torch.zeros(
            n_nodes, flow_features.size(1), device=device, dtype=flow_features.dtype
        )
        counts = torch.zeros(n_nodes, 1, device=device, dtype=flow_features.dtype)
        ones = torch.ones(n_flows, 1, device=device, dtype=flow_features.dtype)
        for node_ids in (src, dst, proto):
            node_features.scatter_add_(
                0,
                node_ids.unsqueeze(1).expand(-1, flow_features.size(1)),
                flow_features,
            )
            counts.scatter_add_(0, node_ids.unsqueeze(1), ones)
        node_features = node_features / counts.clamp(min=1.0)

        # Corrected bidirectional relations.
        comm_src = torch.cat([src, dst], dim=0)
        comm_dst = torch.cat([dst, src], dim=0)
        proto_src = torch.cat([src, proto], dim=0)
        proto_dst = torch.cat([proto, src], dim=0)

        if self.variant.graph_mode == "hetero":
            relations = [(comm_src, comm_dst), (proto_src, proto_dst)]
        else:
            relations = [
                (
                    torch.cat([comm_src, proto_src], dim=0),
                    torch.cat([comm_dst, proto_dst], dim=0),
                )
            ]

        node_embeddings = self.graph_encoder(node_features, relations)
        gate = torch.sigmoid(self.context_gate)
        context = gate * (
            node_embeddings[src] + node_embeddings[dst] + node_embeddings[proto]
        ) / 3.0
        detector_input = torch.cat([flow_features, context], dim=1)
        return detector_input, labels


In [7]:

# Cell 7 — Losses, threshold calibration, metrics, and score collection

def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
    gamma: float,
    pos_weight: torch.Tensor,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
        pos_weight=pos_weight,
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0 - alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1 - pt).pow(gamma) * bce).mean()


def safe_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    result = np.zeros_like(numerator, dtype=np.float64)
    mask = denominator > 0
    result[mask] = numerator[mask] / denominator[mask]
    return result


def find_best_threshold_macro_f1_exact(scores: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(scores) == 0 or len(np.unique(labels)) < 2:
        return 0.0, float(np.median(scores)) if len(scores) else 0.5

    order = np.argsort(scores)
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_ends = np.r_[np.where(np.diff(sorted_scores) != 0)[0], len(sorted_scores) - 1]

    cumulative_positive = np.cumsum(sorted_labels)[unique_ends]
    count_le = unique_ends + 1
    cumulative_negative = count_le - cumulative_positive
    total_positive = int(sorted_labels.sum())
    total_negative = int(len(sorted_labels) - total_positive)

    tp = total_positive - cumulative_positive
    fp = total_negative - cumulative_negative
    fn = cumulative_positive
    tn = cumulative_negative

    f1_positive = safe_divide(2 * tp, 2 * tp + fp + fn)
    f1_negative = safe_divide(2 * tn, 2 * tn + fn + fp)
    macro_f1 = 0.5 * (f1_positive + f1_negative)

    # Candidate: all samples predicted positive.
    all_positive_macro = 0.5 * (
        (2 * total_positive) / max(2 * total_positive + total_negative, 1)
    )

    best_index = int(np.argmax(macro_f1))
    best_f1 = float(macro_f1[best_index])
    best_threshold = float(sorted_scores[unique_ends[best_index]])

    if all_positive_macro > best_f1:
        best_f1 = float(all_positive_macro)
        best_threshold = float(np.nextafter(sorted_scores[0], -np.inf))

    return best_f1, best_threshold


@torch.no_grad()
def collect_scores(
    model: THATIDSVariantModel,
    windows: List[GraphWindow],
    device: torch.device,
) -> Dict[str, np.ndarray]:
    model.eval()
    scores, labels_all, probabilities_all, reconstruction_all = [], [], [], []

    for window in windows:
        detector_input, labels = model.prepare_flow_input(window, device)
        _, reconstruction, logits = model.detector(detector_input)

        probability = torch.sigmoid(logits) if logits is not None else None
        reconstruction_error = (
            model.detector.reconstruction_error(detector_input, reconstruction)
            if reconstruction is not None else None
        )

        mode = model.variant.inference_mode
        if mode == "hybrid":
            if probability is None or reconstruction_error is None:
                raise RuntimeError("Hybrid inference requires classifier and decoder")
            score = (
                SCORE_CLS_WEIGHT * probability
                + SCORE_RECON_WEIGHT * torch.log1p(reconstruction_error)
            )
        elif mode == "classifier":
            if probability is None:
                raise RuntimeError("Classifier inference requires classifier head")
            score = probability
        elif mode == "reconstruction":
            if reconstruction_error is None:
                raise RuntimeError("Reconstruction inference requires decoder")
            score = reconstruction_error
        else:
            raise ValueError(f"Unknown inference mode: {mode}")

        scores.append(score.detach().cpu().numpy().astype(np.float32, copy=False))
        labels_all.append(labels.detach().cpu().numpy().astype(np.int8, copy=False))
        probabilities_all.append(
            probability.detach().cpu().numpy().astype(np.float32, copy=False)
            if probability is not None
            else np.full(len(labels), np.nan, dtype=np.float32)
        )
        reconstruction_all.append(
            reconstruction_error.detach().cpu().numpy().astype(np.float32, copy=False)
            if reconstruction_error is not None
            else np.full(len(labels), np.nan, dtype=np.float32)
        )

    return {
        "scores": np.concatenate(scores),
        "labels": np.concatenate(labels_all).astype(np.int64, copy=False),
        "probabilities": np.concatenate(probabilities_all),
        "reconstruction_errors": np.concatenate(reconstruction_all),
    }


def evaluate_scores(
    collected: Dict[str, np.ndarray],
    threshold: Optional[float] = None,
    score_sign: Optional[int] = None,
) -> Dict[str, Any]:
    raw_scores = collected["scores"].astype(np.float64, copy=False)
    labels = collected["labels"].astype(np.int64, copy=False)

    if len(np.unique(labels)) < 2:
        raise ValueError("Evaluation split must contain both benign and attack labels")

    raw_auc = roc_auc_score(labels, raw_scores)
    if score_sign is None:
        score_sign = -1 if raw_auc < 0.5 else 1
    oriented_scores = score_sign * raw_scores

    auc = roc_auc_score(labels, oriented_scores)
    pr_auc = average_precision_score(labels, oriented_scores)

    if threshold is None:
        f1_macro, threshold = find_best_threshold_macro_f1_exact(oriented_scores, labels)
    else:
        predictions = (oriented_scores > threshold).astype(np.int8)
        f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)

    predictions = (oriented_scores > threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

    return {
        "f1_macro": float(f1_macro),
        "auc_roc": float(auc),
        "pr_auc": float(pr_auc),
        "threshold": float(threshold),
        "score_sign": int(score_sign),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": float((tp + tn) / max(tp + tn + fp + fn, 1)),
        "precision": float(tp / max(tp + fp, 1)),
        "recall": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_accuracy": float(
            0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
        ),
        "f1_attack": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "f1_benign": float(2 * tn / max(2 * tn + fn + fp, 1)),
        "fp_rate": float(fp / max(fp + tn, 1)),
        "tp_rate": float(tp / max(tp + fn, 1)),
        "scores": oriented_scores.astype(np.float32),
        "predictions": predictions,
    }


In [8]:

# Cell 8 — Checkpointed training and per-task execution

def train_one_epoch(
    model: THATIDSVariantModel,
    windows: List[GraphWindow],
    optimizer: torch.optim.Optimizer,
    positive_class_weight: float,
) -> Dict[str, float]:
    model.train()
    totals = {"loss": 0.0, "reconstruction_loss": 0.0, "classification_loss": 0.0}
    batches = 0
    pos_weight = torch.tensor([positive_class_weight], device=DEVICE, dtype=torch.float32)

    for window in windows:
        detector_input, labels = model.prepare_flow_input(window, DEVICE)
        _, reconstruction, logits = model.detector(detector_input)
        y = labels.float()

        components = []
        reconstruction_loss = torch.zeros((), device=DEVICE)
        classification_loss = torch.zeros((), device=DEVICE)

        if model.variant.train_reconstruction:
            benign_mask = labels == 0
            if benign_mask.any():
                reconstruction_errors = model.detector.reconstruction_error(
                    detector_input[benign_mask], reconstruction[benign_mask]
                )
                reconstruction_loss = reconstruction_errors.mean()
                components.append(LAMBDA_RECON * reconstruction_loss)

        if model.variant.train_classifier:
            classification_loss = focal_bce_with_logits(
                logits,
                y,
                alpha=FOCAL_ALPHA,
                gamma=FOCAL_GAMMA,
                pos_weight=pos_weight,
            )
            components.append(LAMBDA_CLS * classification_loss)

        if not components:
            raise RuntimeError(f"Variant {model.variant.name} has no trainable objective")

        loss = sum(components)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        totals["loss"] += float(loss.detach().item())
        totals["reconstruction_loss"] += float(reconstruction_loss.detach().item())
        totals["classification_loss"] += float(classification_loss.detach().item())
        batches += 1

    return {key: value / max(batches, 1) for key, value in totals.items()}


def load_history(history_path: Path) -> List[Dict[str, Any]]:
    if not history_path.exists():
        return []
    return pd.read_csv(history_path).to_dict(orient="records")


def save_history(history: List[Dict[str, Any]], history_path: Path) -> None:
    history_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = history_path.with_suffix(".csv.tmp")
    pd.DataFrame(history).to_csv(tmp, index=False)
    os.replace(tmp, history_path)


def run_variant_task(
    bundle: DatasetBundle,
    seed: int,
    variant: VariantConfig,
    train_windows: List[GraphWindow],
    val_windows: List[GraphWindow],
    test_windows: List[GraphWindow],
    test_idx: np.ndarray,
    positive_class_weight: float,
    dataset_dir: Path,
    protocol_name: str,
    split_fingerprint: str,
) -> Dict[str, Any]:
    task_dir = dataset_dir / variant.name / f"seed_{seed}"
    task_dir.mkdir(parents=True, exist_ok=True)

    completed_path = task_dir / "completed.json"
    best_path = task_dir / "best.pt"
    last_path = task_dir / "last.pt"
    history_path = task_dir / "history.csv"
    result_path = task_dir / "test_metrics.json"
    prediction_path = task_dir / "test_predictions.npz"
    config_path = task_dir / "config.json"

    expected_fingerprint = task_config_fingerprint(
        dataset_name=bundle.name,
        protocol_name=protocol_name,
        seed=seed,
        feature_names=bundle.feature_names,
        variant=variant,
        split_fingerprint=split_fingerprint,
    )

    if completed_path.exists():
        with open(completed_path, "r", encoding="utf-8") as f:
            completed = json.load(f)
        if (
            completed.get("config_fingerprint") == expected_fingerprint
            and result_path.exists()
            and prediction_path.exists()
        ):
            print(f"  SKIP completed: {bundle.name} | {variant.name} | seed={seed}")
            with open(result_path, "r", encoding="utf-8") as f:
                return json.load(f)
        if completed.get("config_fingerprint") == expected_fingerprint:
            print("  completed.json exists but result/prediction is missing; resuming task.")
            completed_path.unlink(missing_ok=True)
        else:
            raise RuntimeError(
                f"Task {task_dir} tồn tại nhưng config fingerprint khác. "
                "Hãy đổi EXPERIMENT_TAG hoặc xóa thư mục task cũ."
            )
    seed_everything(seed)
    model = THATIDSVariantModel(flow_dim=len(bundle.feature_names), variant=variant).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    task_config = {
        "dataset": bundle.name,
        "protocol": protocol_name,
        "split_fingerprint": split_fingerprint,
        "seed": seed,
        "variant": asdict(variant),
        "parameters": parameter_count,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "window_size": WINDOW_SIZE,
        "feature_names": bundle.feature_names,
        "config_fingerprint": expected_fingerprint,
    }
    atomic_json_dump(task_config, config_path)

    start_epoch = 1
    best_val_f1 = -1.0
    best_epoch = -1
    best_threshold = 0.5
    best_score_sign = 1
    patience_counter = 0
    history = load_history(history_path)

    if last_path.exists():
        checkpoint = load_torch_checkpoint(last_path, map_location=DEVICE)
        if checkpoint.get("config_fingerprint") != expected_fingerprint:
            raise RuntimeError(f"Checkpoint config mismatch: {last_path}")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val_f1 = float(checkpoint["best_val_f1"])
        best_epoch = int(checkpoint["best_epoch"])
        best_threshold = float(checkpoint["best_threshold"])
        best_score_sign = int(checkpoint["best_score_sign"])
        patience_counter = int(checkpoint["patience_counter"])
        restore_rng_state(checkpoint.get("rng_state"))
        print(f"  RESUME {variant.name} seed={seed} from epoch {start_epoch}")
    else:
        print(f"  START {variant.name} seed={seed} | params={parameter_count:,}")

    task_start = time.time()
    if patience_counter >= PATIENCE:
        print(
            f"  Early-stopping condition was already reached in last.pt "
            f"(best epoch={best_epoch}); skip directly to test evaluation."
        )

    for epoch in range(start_epoch, MAX_EPOCHS + 1) if patience_counter < PATIENCE else []:
        epoch_start = time.time()
        train_metrics = train_one_epoch(
            model,
            train_windows,
            optimizer,
            positive_class_weight,
        )

        validation_collected = collect_scores(model, val_windows, DEVICE)
        validation_metrics = evaluate_scores(validation_collected)
        gate = model.gate_value()

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_reconstruction_loss": train_metrics["reconstruction_loss"],
            "train_classification_loss": train_metrics["classification_loss"],
            "val_f1_macro": validation_metrics["f1_macro"],
            "val_auc_roc": validation_metrics["auc_roc"],
            "val_pr_auc": validation_metrics["pr_auc"],
            "val_threshold": validation_metrics["threshold"],
            "val_score_sign": validation_metrics["score_sign"],
            "gate": np.nan if gate is None else gate,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - epoch_start,
        }
        history = [h for h in history if int(h["epoch"]) != epoch]
        history.append(row)
        history.sort(key=lambda x: int(x["epoch"]))
        save_history(history, history_path)

        improved = validation_metrics["f1_macro"] > best_val_f1
        if improved:
            best_val_f1 = validation_metrics["f1_macro"]
            best_epoch = epoch
            best_threshold = validation_metrics["threshold"]
            best_score_sign = validation_metrics["score_sign"]
            patience_counter = 0
            atomic_torch_save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "best_val_f1": best_val_f1,
                    "best_threshold": best_threshold,
                    "best_score_sign": best_score_sign,
                    "variant": asdict(variant),
                    "config_fingerprint": expected_fingerprint,
                },
                best_path,
            )
        else:
            patience_counter += 1

        if epoch % SAVE_LAST_EVERY == 0 or improved or epoch == MAX_EPOCHS:
            atomic_torch_save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_val_f1": best_val_f1,
                    "best_epoch": best_epoch,
                    "best_threshold": best_threshold,
                    "best_score_sign": best_score_sign,
                    "patience_counter": patience_counter,
                    "rng_state": capture_rng_state(),
                    "variant": asdict(variant),
                    "config_fingerprint": expected_fingerprint,
                },
                last_path,
            )

        gate_text = "NA" if gate is None else f"{gate:.4f}"
        print(
            f"    Ep {epoch:03d}/{MAX_EPOCHS} | "
            f"loss={train_metrics['loss']:.5f} | "
            f"valF1={validation_metrics['f1_macro']:.4f} | "
            f"AUC={validation_metrics['auc_roc']:.4f} | "
            f"PR={validation_metrics['pr_auc']:.4f} | "
            f"thr={validation_metrics['threshold']:.4f} | "
            f"gate={gate_text} | pat={patience_counter}/{PATIENCE} | "
            f"{time.time() - epoch_start:.0f}s"
        )

        if patience_counter >= PATIENCE:
            print(f"    Early stop. Best epoch={best_epoch}, val F1={best_val_f1:.4f}")
            break

    if not best_path.exists():
        raise RuntimeError(f"Best checkpoint was not created: {best_path}")

    best_checkpoint = load_torch_checkpoint(best_path, map_location=DEVICE)
    model.load_state_dict(best_checkpoint["model_state_dict"])

    test_collected = collect_scores(model, test_windows, DEVICE)
    test_metrics = evaluate_scores(
        test_collected,
        threshold=best_threshold,
        score_sign=best_score_sign,
    )

    # Scores follow the same order as test_idx and the graph batches.
    test_attack_codes = bundle.attack_codes[test_idx][:len(test_metrics["scores"])]
    per_attack = []
    for code, attack_name in enumerate(bundle.attack_classes):
        mask = (test_attack_codes == code) & (test_collected["labels"] == 1)
        count = int(mask.sum())
        if count == 0:
            continue
        detected = int(test_metrics["predictions"][mask].sum())
        per_attack.append({
            "attack": str(attack_name),
            "count": count,
            "detected": detected,
            "recall": float(detected / count),
        })

    result = {
        "dataset": bundle.name,
        "protocol": protocol_name,
        "split_fingerprint": split_fingerprint,
        "variant": variant.name,
        "edge_control": variant.edge_control,
        "seed": seed,
        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "validation": {
            "f1_macro": best_val_f1,
            "threshold": best_threshold,
            "score_sign": best_score_sign,
        },
        "test": {
            key: value
            for key, value in test_metrics.items()
            if key not in {"scores", "predictions"}
        },
        "gate_final": model.gate_value(),
        "positive_class_weight": positive_class_weight,
        "per_attack": per_attack,
        "elapsed_seconds_current_session": time.time() - task_start,
        "elapsed_seconds_history": float(
            sum(float(row.get("epoch_seconds", 0.0)) for row in history)
        ),
        "epochs_recorded": int(len(history)),
        "config_fingerprint": expected_fingerprint,
    }
    atomic_json_dump(result, result_path)

    np.savez_compressed(
        prediction_path,
        scores=test_metrics["scores"].astype(np.float32),
        predictions=test_metrics["predictions"].astype(np.int8),
        labels=test_collected["labels"].astype(np.int8),
        probabilities=test_collected["probabilities"].astype(np.float32),
        reconstruction_errors=test_collected["reconstruction_errors"].astype(np.float32),
        attack_codes=test_attack_codes.astype(np.int16),
        attack_classes=bundle.attack_classes.astype(str),
    )

    atomic_json_dump(
        {
            "completed": True,
            "dataset": bundle.name,
            "protocol": protocol_name,
            "variant": variant.name,
            "seed": seed,
            "completed_at_unix": time.time(),
            "config_fingerprint": expected_fingerprint,
        },
        completed_path,
    )

    print(
        f"  DONE {variant.name} seed={seed} | "
        f"F1={result['test']['f1_macro']:.4f} | "
        f"AUC={result['test']['auc_roc']:.4f} | "
        f"PR={result['test']['pr_auc']:.4f} | "
        f"FPR={100 * result['test']['fp_rate']:.4f}%"
    )

    del model, optimizer, test_collected, test_metrics
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


In [10]:
def _cpu_byte_tensor(value):
    """Đưa RNG state về CPU ByteTensor đúng định dạng PyTorch yêu cầu."""
    if isinstance(value, torch.Tensor):
        return (
            value.detach()
            .to(device="cpu", dtype=torch.uint8)
            .contiguous()
        )

    return torch.as_tensor(
        value,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


def restore_rng_state(state):
    """Khôi phục RNG state an toàn khi resume trên CPU hoặc GPU."""
    if not state:
        return

    if "python" in state:
        random.setstate(state["python"])

    if "numpy" in state:
        np.random.set_state(state["numpy"])

    if "torch" in state:
        torch.set_rng_state(
            _cpu_byte_tensor(state["torch"])
        )

    if torch.cuda.is_available() and "cuda" in state:
        cuda_states = state["cuda"]

        if isinstance(cuda_states, torch.Tensor):
            cuda_states = [cuda_states]

        for device_index, cuda_state in enumerate(
            list(cuda_states)[:torch.cuda.device_count()]
        ):
            torch.cuda.set_rng_state(
                _cpu_byte_tensor(cuda_state),
                device=device_index,
            )

print("RNG resume hotfix loaded.")

RNG resume hotfix loaded.


In [11]:
# Cell 9 — Run paper-mode audit and training

def copy_small_audit_files(cache_dir: Path, dataset_dir: Path) -> None:
    for name in [
        "duplicate_audit_summary.json",
        "duplicate_audit_per_attack.csv",
        "model_feature_names.csv",
    ]:
        source = cache_dir / name
        destination = dataset_dir / name
        if source.exists():
            shutil.copy2(source, destination)


def run_dataset_step4(dataset_name: str) -> None:
    csv_path = resolve_dataset_path(dataset_name)
    dataset_dir = OUTPUT_ROOT / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = resolve_dataset_cache_dir(dataset_name)

    bundle = load_dataset_bundle(dataset_name, csv_path)

    manifest = {
        "dataset": dataset_name,
        "csv_path": csv_path,
        "dataset_cache_dir": str(cache_dir),
        "experiment_tag": EXPERIMENT_TAG,
        "group_split_version": GROUP_SPLIT_VERSION,
        "run_mode": RUN_MODE,
        "seeds": SEEDS,
        "models": MODELS_TO_RUN,
        "core_models": CORE_MODELS,
        "run_shuffled_edge_control": RUN_SHUFFLED_EDGE_CONTROL,
        "train_splits": TRAIN_SPLITS,
        "feature_names": bundle.feature_names,
        "attack_classes": bundle.attack_classes.tolist(),
        "important_notes": [
            "Exact feature groups are computed on normalized float32 model-input vectors before MinMax scaling.",
            "Scaler is fitted independently on each seed's exact-group-disjoint train split.",
            "Graph batches are fixed-size, batch-local, non-temporal graphs in shuffled split order.",
            "full_shuffled_edges preserves destination/protocol assignment frequencies within each batch but randomizes their association with flows.",
            "graph_classifier is an internal ablation, not an official E-GraphSAGE reproduction.",
        ],
    }
    atomic_json_dump(manifest, dataset_dir / "experiment_manifest.json")

    hashes = compute_or_load_feature_hashes(bundle, cache_dir)
    unique_hashes, group_inverse, group_counts = compute_or_load_feature_groups(
        hashes, cache_dir
    )

    run_duplicate_audit(
        bundle,
        hashes,
        group_inverse,
        group_counts,
        cache_dir,
    )
    copy_small_audit_files(cache_dir, dataset_dir)

    protocols_to_prepare = set(TRAIN_SPLITS)
    if AUDIT_RANDOM_PROTOCOL:
        protocols_to_prepare.add("random")

    for protocol_name in sorted(protocols_to_prepare):
        for seed in SEEDS:
            print("\n" + "=" * 100)
            print(f"DATASET={dataset_name} | PROTOCOL={protocol_name} | SEED={seed}")
            print("=" * 100)

            protocol_dir = dataset_dir / protocol_name
            shared_dir = protocol_dir / "shared" / f"seed_{seed}"

            train_idx, val_idx, test_idx, split_metadata = get_or_create_protocol_split(
                bundle=bundle,
                protocol_name=protocol_name,
                seed=seed,
                unique_hashes=unique_hashes,
                group_inverse=group_inverse,
                group_counts=group_counts,
                shared_dir=shared_dir,
            )

            if not RUN_TRAINING or protocol_name not in TRAIN_SPLITS:
                print("Audit only; model training skipped for", protocol_name)
                continue

            split_fingerprint = split_metadata["split_fingerprint"]
            scaler = fit_or_load_scaler(
                bundle,
                train_idx,
                shared_dir,
                split_fingerprint=split_fingerprint,
            )

            y_train = bundle.labels[train_idx]
            n_positive = int((y_train == 1).sum())
            n_negative = int((y_train == 0).sum())
            positive_class_weight = min(
                n_negative / max(n_positive, 1),
                POS_WEIGHT_CAP,
            )

            print(
                f"Train={len(train_idx):,} | Validation={len(val_idx):,} | "
                f"Test={len(test_idx):,} | pos_weight={positive_class_weight:.4f}"
            )

            print("Building graph batches shared by all variants...")
            train_windows = prepare_graph_windows(
                bundle,
                train_idx,
                scaler,
                "Train",
                control_seed=seed * 10_000 + 101,
            )
            val_windows = prepare_graph_windows(
                bundle,
                val_idx,
                scaler,
                "Validation",
                control_seed=seed * 10_000 + 103,
            )
            test_windows = prepare_graph_windows(
                bundle,
                test_idx,
                scaler,
                "Test",
                control_seed=seed * 10_000 + 107,
            )

            for model_name in MODELS_TO_RUN:
                variant = EXPERIMENT_VARIANTS[model_name]
                run_variant_task(
                    bundle=bundle,
                    seed=seed,
                    variant=variant,
                    train_windows=train_windows,
                    val_windows=val_windows,
                    test_windows=test_windows,
                    test_idx=test_idx,
                    positive_class_weight=positive_class_weight,
                    dataset_dir=protocol_dir,
                    protocol_name=protocol_name,
                    split_fingerprint=split_fingerprint,
                )

            del train_windows, val_windows, test_windows, scaler
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del hashes, unique_hashes, group_inverse, group_counts, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


for dataset_name in DATASETS_TO_RUN:
    run_dataset_step4(dataset_name)

print("\nAll requested paper-mode tasks completed or resumed from checkpoints.")


Reuse pilot feature cache for UNSW_V2: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_v1/pilot/UNSW_V2

Loading UNSW_V2: /content/drive/MyDrive/NF-UNSW-NB15-v2.csv
Rows=2,390,275 | model-input features=39
  Converted features: 10/39
  Converted features: 20/39
  Converted features: 30/39
  Converted features: 39/39
Loaded in 0.2 min
Label counts: {np.int8(0): np.int64(2295222), np.int8(1): np.int64(95053)}
Attack classes: 10
Loaded feature hashes: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_v1/pilot/UNSW_V2/exact_feature_hashes_uint64.npy
Loaded feature-group cache: 236,999 groups
Loaded duplicate audit: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_v1/pilot/UNSW_V2/duplicate_audit_summary.json

DATASET=UNSW_V2 | PROTOCOL=exact_feature_group | SEED=42
Loaded split: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_paper_v2/paper/UNSW_V2/exact_feature_group/shared/seed_42/spl

In [12]:
# Cell 10 — Robust aggregation, completeness audit, and paper-ready tables

def sample_std(values: pd.Series) -> float:
    return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0


def discovered_datasets() -> List[str]:
    names = []
    for path in OUTPUT_ROOT.iterdir():
        if path.is_dir() and path.name in DATA_PATH_CANDIDATES:
            names.append(path.name)
    return sorted(names)


def discovered_seed_values(shared_root: Path) -> List[int]:
    seeds = []
    if not shared_root.exists():
        return seeds
    for path in shared_root.glob("seed_*"):
        try:
            seeds.append(int(path.name.split("_", 1)[1]))
        except Exception:
            pass
    return sorted(set(seeds))


def load_step4_outputs():
    audit_rows, run_rows, per_attack_rows, balance_rows = [], [], [], []

    for dataset_name in discovered_datasets():
        dataset_dir = OUTPUT_ROOT / dataset_name
        duplicate_path = dataset_dir / "duplicate_audit_summary.json"
        duplicate = {}
        if duplicate_path.exists():
            with open(duplicate_path, "r", encoding="utf-8") as f:
                duplicate = json.load(f)

        for protocol_name in ["random", "exact_feature_group"]:
            protocol_dir = dataset_dir / protocol_name
            seeds_found = discovered_seed_values(protocol_dir / "shared")
            for seed in seeds_found:
                shared_dir = protocol_dir / "shared" / f"seed_{seed}"
                overlap_path = shared_dir / "split_overlap_audit.json"
                if overlap_path.exists():
                    with open(overlap_path, "r", encoding="utf-8") as f:
                        overlap = json.load(f)
                    audit_rows.append({
                        "dataset": dataset_name,
                        "protocol": protocol_name,
                        "seed": seed,
                        "repeated_rows_rate": duplicate.get("repeated_rows_rate", np.nan),
                        **overlap,
                    })

                balance_path = shared_dir / "split_attack_distribution.csv"
                if balance_path.exists():
                    frame = pd.read_csv(balance_path)
                    frame.insert(0, "seed", seed)
                    frame.insert(0, "protocol", protocol_name)
                    frame.insert(0, "dataset", dataset_name)
                    balance_rows.extend(frame.to_dict(orient="records"))

            if not protocol_dir.exists():
                continue
            model_dirs = [
                path for path in protocol_dir.iterdir()
                if path.is_dir() and path.name != "shared"
            ]
            for model_dir in model_dirs:
                model_name = model_dir.name
                for seed_dir in model_dir.glob("seed_*"):
                    try:
                        seed = int(seed_dir.name.split("_", 1)[1])
                    except Exception:
                        continue
                    result_path = seed_dir / "test_metrics.json"
                    if not result_path.exists():
                        continue
                    with open(result_path, "r", encoding="utf-8") as f:
                        result = json.load(f)
                    test = result["test"]
                    row = {
                        "dataset": dataset_name,
                        "protocol": protocol_name,
                        "model": model_name,
                        "edge_control": result.get("edge_control", "observed"),
                        "seed": seed,
                        "parameters": result["parameters"],
                        "best_epoch": result["best_epoch"],
                        "threshold": result["validation"]["threshold"],
                        "gate_final": result.get("gate_final", np.nan),
                        "elapsed_seconds_history": result.get("elapsed_seconds_history", np.nan),
                    }
                    for metric in [
                        "f1_macro", "auc_roc", "pr_auc", "accuracy", "precision",
                        "recall", "specificity", "balanced_accuracy", "f1_attack",
                        "f1_benign", "fp_rate", "tp_rate", "tn", "fp", "fn", "tp",
                    ]:
                        row[metric] = test.get(metric, np.nan)
                    run_rows.append(row)

                    for item in result.get("per_attack", []):
                        per_attack_rows.append({
                            "dataset": dataset_name,
                            "protocol": protocol_name,
                            "model": model_name,
                            "seed": seed,
                            "attack": item["attack"],
                            "count": item["count"],
                            "detected": item["detected"],
                            "recall": item["recall"],
                        })

    return (
        pd.DataFrame(audit_rows),
        pd.DataFrame(run_rows),
        pd.DataFrame(per_attack_rows),
        pd.DataFrame(balance_rows),
    )


def build_completeness_table() -> pd.DataFrame:
    rows = []
    for dataset_name in PAPER_DATASETS:
        for seed in SEEDS:
            for model_name in MODELS_TO_RUN:
                task_dir = (
                    OUTPUT_ROOT / dataset_name / "exact_feature_group"
                    / model_name / f"seed_{seed}"
                )
                result_path = task_dir / "test_metrics.json"
                prediction_path = task_dir / "test_predictions.npz"
                completed_path = task_dir / "completed.json"
                rows.append({
                    "dataset": dataset_name,
                    "seed": seed,
                    "model": model_name,
                    "completed": bool(
                        result_path.exists()
                        and prediction_path.exists()
                        and completed_path.exists()
                    ),
                    "has_result": result_path.exists(),
                    "has_predictions": prediction_path.exists(),
                    "has_completed_marker": completed_path.exists(),
                    "task_dir": str(task_dir),
                })
    return pd.DataFrame(rows)


audit_df, all_runs_df, per_attack_df, split_balance_df = load_step4_outputs()
completeness_df = build_completeness_table()
completeness_df.to_csv(OUTPUT_ROOT / "paper_run_completeness.csv", index=False)

completed_count = int(completeness_df["completed"].sum())
expected_count = int(len(completeness_df))
print(f"Paper task completeness: {completed_count}/{expected_count}")
if completed_count < expected_count:
    print("Missing tasks:")
    display(completeness_df.loc[~completeness_df["completed"], ["dataset", "seed", "model"]])
    if STRICT_FINAL_COMPLETENESS:
        raise RuntimeError("Paper run is incomplete. Resume missing tasks before finalizing tables.")

if not audit_df.empty:
    audit_df = audit_df.sort_values(["dataset", "protocol", "seed"])
    audit_df.to_csv(OUTPUT_ROOT / "protocol_overlap_audit_all.csv", index=False)
    audit_summary = (
        audit_df.groupby(["dataset", "protocol"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            repeated_rows_rate=("repeated_rows_rate", "mean"),
            validation_feature_overlap_rate_mean=("validation_feature_overlap_rate", "mean"),
            validation_feature_overlap_rate_max=("validation_feature_overlap_rate", "max"),
            test_feature_overlap_rate_mean=("test_feature_overlap_rate", "mean"),
            test_feature_overlap_rate_max=("test_feature_overlap_rate", "max"),
            unique_groups_shared_train_test_max=("unique_groups_shared_train_test", "max"),
            unique_groups_spanning_all_three_max=("unique_groups_spanning_all_three", "max"),
        )
    )
    audit_summary.to_csv(OUTPUT_ROOT / "protocol_overlap_audit_summary.csv", index=False)
    print("=== Protocol overlap audit ===")
    display(audit_summary)
else:
    audit_summary = pd.DataFrame()

if not split_balance_df.empty:
    split_balance_df.to_csv(OUTPUT_ROOT / "split_attack_distribution_all_seeds.csv", index=False)
    exact_balance = split_balance_df[
        split_balance_df["protocol"] == "exact_feature_group"
    ].copy()
    balance_summary = (
        exact_balance.groupby(["dataset", "attack", "split"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            family_total_mean=("family_total", "mean"),
            fraction_of_attack_family_mean=("fraction_of_attack_family", "mean"),
            fraction_of_attack_family_std=("fraction_of_attack_family", sample_std),
            absolute_fraction_error_mean=("absolute_fraction_error", "mean"),
            absolute_fraction_error_max=("absolute_fraction_error", "max"),
        )
    )
    balance_summary.to_csv(OUTPUT_ROOT / "split_attack_balance_summary.csv", index=False)
    worst_family_balance = (
        exact_balance.sort_values("absolute_fraction_error", ascending=False)
        .groupby(["dataset", "seed"], as_index=False)
        .head(10)
        .sort_values(["dataset", "seed", "absolute_fraction_error"], ascending=[True, True, False])
    )
    worst_family_balance.to_csv(OUTPUT_ROOT / "split_attack_balance_worst_cases.csv", index=False)
else:
    balance_summary = pd.DataFrame()
    worst_family_balance = pd.DataFrame()

if not all_runs_df.empty:
    all_runs_df = all_runs_df.sort_values(["dataset", "protocol", "model", "seed"])
    all_runs_df.to_csv(OUTPUT_ROOT / "exact_group_all_runs.csv", index=False)

    exact_runs = all_runs_df[
        all_runs_df["protocol"] == "exact_feature_group"
    ].copy()
    summary = (
        exact_runs.groupby(["dataset", "protocol", "model", "edge_control"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            parameters=("parameters", "mean"),
            best_epoch_mean=("best_epoch", "mean"),
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_std=("f1_macro", sample_std),
            auc_roc_mean=("auc_roc", "mean"),
            auc_roc_std=("auc_roc", sample_std),
            pr_auc_mean=("pr_auc", "mean"),
            pr_auc_std=("pr_auc", sample_std),
            fp_rate_mean=("fp_rate", "mean"),
            fp_rate_std=("fp_rate", sample_std),
            tp_rate_mean=("tp_rate", "mean"),
            tp_rate_std=("tp_rate", sample_std),
            mean_fp=("fp", "mean"),
            mean_fn=("fn", "mean"),
            gate_mean=("gate_final", "mean"),
            gate_std=("gate_final", sample_std),
            elapsed_hours_mean=("elapsed_seconds_history", lambda x: float(np.nanmean(x)) / 3600.0),
        )
    )
    summary.to_csv(OUTPUT_ROOT / "exact_group_model_summary.csv", index=False)
    print("=== Exact-feature-group model summary ===")
    display(summary)

    paper_table = summary.copy()
    for metric in ["f1_macro", "auc_roc", "pr_auc", "fp_rate", "tp_rate"]:
        paper_table[metric] = paper_table.apply(
            lambda row: f"{row[f'{metric}_mean']:.6f} ± {row[f'{metric}_std']:.6f}",
            axis=1,
        )
    paper_table = paper_table[[
        "dataset", "model", "n_seeds", "parameters",
        "f1_macro", "auc_roc", "pr_auc", "fp_rate", "tp_rate",
    ]]
    paper_table.to_csv(OUTPUT_ROOT / "paper_main_results_table.csv", index=False)
    with open(OUTPUT_ROOT / "paper_main_results_table.tex", "w", encoding="utf-8") as f:
        f.write(paper_table.to_latex(index=False, escape=False))

    paired_rows = []
    for (dataset_name, protocol_name, seed), group in exact_runs.groupby(
        ["dataset", "protocol", "seed"]
    ):
        by_model = group.set_index("model")
        if "full" not in by_model.index:
            continue
        for reference in [
            "mlp", "no_graph", "graph_classifier", "ae_only", "full_shuffled_edges"
        ]:
            if reference not in by_model.index:
                continue
            paired_rows.append({
                "dataset": dataset_name,
                "protocol": protocol_name,
                "seed": seed,
                "comparison": f"full_minus_{reference}",
                "delta_f1_macro": float(by_model.loc["full", "f1_macro"] - by_model.loc[reference, "f1_macro"]),
                "delta_auc_roc": float(by_model.loc["full", "auc_roc"] - by_model.loc[reference, "auc_roc"]),
                "delta_pr_auc": float(by_model.loc["full", "pr_auc"] - by_model.loc[reference, "pr_auc"]),
                "delta_fp_rate": float(by_model.loc["full", "fp_rate"] - by_model.loc[reference, "fp_rate"]),
                "delta_tp_rate": float(by_model.loc["full", "tp_rate"] - by_model.loc[reference, "tp_rate"]),
                "delta_fn": float(by_model.loc["full", "fn"] - by_model.loc[reference, "fn"]),
                "delta_fp": float(by_model.loc["full", "fp"] - by_model.loc[reference, "fp"]),
            })

    paired_df = pd.DataFrame(paired_rows)
    if not paired_df.empty:
        paired_df.to_csv(OUTPUT_ROOT / "exact_group_paired_model_differences.csv", index=False)
        paired_summary = (
            paired_df.groupby(["dataset", "protocol", "comparison"], as_index=False)
            .agg(
                n_seeds=("seed", "nunique"),
                delta_f1_macro_mean=("delta_f1_macro", "mean"),
                delta_f1_macro_std=("delta_f1_macro", sample_std),
                delta_f1_macro_min=("delta_f1_macro", "min"),
                delta_f1_macro_max=("delta_f1_macro", "max"),
                f1_wins=("delta_f1_macro", lambda x: int((x > 0).sum())),
                delta_auc_roc_mean=("delta_auc_roc", "mean"),
                delta_pr_auc_mean=("delta_pr_auc", "mean"),
                delta_pr_auc_std=("delta_pr_auc", sample_std),
                pr_auc_wins=("delta_pr_auc", lambda x: int((x > 0).sum())),
                delta_fp_rate_mean=("delta_fp_rate", "mean"),
                lower_fpr_wins=("delta_fp_rate", lambda x: int((x < 0).sum())),
                delta_tp_rate_mean=("delta_tp_rate", "mean"),
                higher_tpr_wins=("delta_tp_rate", lambda x: int((x > 0).sum())),
                delta_fn_mean=("delta_fn", "mean"),
                delta_fp_mean=("delta_fp", "mean"),
            )
        )
        paired_summary.to_csv(OUTPUT_ROOT / "exact_group_paired_model_summary.csv", index=False)
        print("=== Paired model differences ===")
        display(paired_summary)
    else:
        paired_summary = pd.DataFrame()
else:
    exact_runs = pd.DataFrame()
    summary = pd.DataFrame()
    paper_table = pd.DataFrame()
    paired_df = pd.DataFrame()
    paired_summary = pd.DataFrame()

if not per_attack_df.empty:
    per_attack_df.to_csv(OUTPUT_ROOT / "exact_group_per_attack_all_runs.csv", index=False)
    per_attack_summary = (
        per_attack_df[
            per_attack_df["protocol"] == "exact_feature_group"
        ]
        .groupby(["dataset", "protocol", "model", "attack"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            support_mean=("count", "mean"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", sample_std),
            recall_min=("recall", "min"),
            recall_max=("recall", "max"),
        )
    )
    per_attack_summary.to_csv(OUTPUT_ROOT / "exact_group_per_attack_summary.csv", index=False)

    def normalize_attack_name(value: str) -> str:
        return value.lower().replace("-", "").replace("_", "").replace(" ", "")

    desired = {
        "UNSW_V2": {"analysis", "dos", "exploits", "generic", "reconnaissance"},
        "CIC_V2": {"infiltration", "infilteration", "bot", "ddosattacksloichttp", "bruteforceweb", "bruteforcexss"},
    }
    hard_rows = []
    for _, row in per_attack_summary.iterrows():
        normalized = normalize_attack_name(str(row["attack"]))
        allowed = {normalize_attack_name(x) for x in desired.get(row["dataset"], set())}
        if normalized in allowed:
            hard_rows.append(row.to_dict())
    hard_class_df = pd.DataFrame(hard_rows)

    if not summary.empty:
        fpr_rows = []
        for _, row in summary.iterrows():
            fpr_rows.append({
                "dataset": row["dataset"],
                "protocol": row["protocol"],
                "model": row["model"],
                "attack": "Benign FPR",
                "n_seeds": row["n_seeds"],
                "support_mean": np.nan,
                "recall_mean": row["fp_rate_mean"],
                "recall_std": row["fp_rate_std"],
                "recall_min": np.nan,
                "recall_max": np.nan,
            })
        hard_class_df = pd.concat([hard_class_df, pd.DataFrame(fpr_rows)], ignore_index=True)

    hard_class_df.to_csv(OUTPUT_ROOT / "conference_hard_class_comparison.csv", index=False)
    print("=== Hard-class diagnostic table ===")
    display(hard_class_df)
else:
    per_attack_summary = pd.DataFrame()
    hard_class_df = pd.DataFrame()

atomic_json_dump(
    {
        "output_root": str(OUTPUT_ROOT),
        "expected_tasks": expected_count,
        "completed_tasks": completed_count,
        "complete": bool(completed_count == expected_count),
        "datasets_discovered": discovered_datasets(),
        "models_expected": MODELS_TO_RUN,
        "seeds_expected": SEEDS,
        "generated_at_unix": time.time(),
    },
    OUTPUT_ROOT / "aggregation_manifest.json",
)


Paper task completeness: 36/36
=== Protocol overlap audit ===


,dataset,protocol,n_seeds,repeated_rows_rate,validation_feature_overlap_rate_mean,validation_feature_overlap_rate_max,test_feature_overlap_rate_mean,test_feature_overlap_rate_max,unique_groups_shared_train_test_max,unique_groups_spanning_all_three_max
0,CIC_V2,exact_feature_group,3,0.524653,0.000000,0.000000,0.000000,0.000000,0,0
1,CIC_V2,random,3,0.524653,0.547994,0.548287,0.548103,0.548241,321192,87687
2,UNSW_V2,exact_feature_group,3,0.900849,0.000000,0.000000,0.000000,0.000000,0,0
3,UNSW_V2,random,3,0.900849,0.934821,0.934920,0.934756,0.934968,58187,20291


=== Exact-feature-group model summary ===


,dataset,protocol,model,edge_control,n_seeds,parameters,best_epoch_mean,f1_macro_mean,f1_macro_std,auc_roc_mean,...,pr_auc_std,fp_rate_mean,fp_rate_std,tp_rate_mean,tp_rate_std,mean_fp,mean_fn,gate_mean,gate_std,elapsed_hours_mean
0,CIC_V2,exact_feature_group,ae_only,observed,3,94027.0,1.000000,0.897003,0.026487,0.943200,...,0.031515,0.043530,0.010583,0.921298,0.063770,142272.000000,33441.000000,0.178354,0.000293,0.141029
1,CIC_V2,exact_feature_group,full,observed,3,157196.0,25.000000,0.987745,0.000948,0.994648,...,0.006887,0.000126,0.000039,0.958478,0.003711,409.000000,18214.000000,0.177238,0.006436,0.443102
2,CIC_V2,exact_feature_group,full_shuffled_edges,shuffled,3,157196.0,29.000000,0.983099,0.007715,0.981735,...,0.021429,0.000324,0.000247,0.944436,0.023692,1057.000000,24933.333333,0.173871,0.009858,0.496930
3,CIC_V2,exact_feature_group,graph_classifier,observed,3,112517.0,19.666667,0.987091,0.000444,0.994077,...,0.005327,0.000126,0.000028,0.956275,0.001659,410.333333,19253.333333,0.192109,0.002317,0.343885
4,CIC_V2,exact_feature_group,mlp,observed,3,70945.0,68.333333,0.984066,0.007223,0.990356,...,0.001988,0.000182,0.000135,0.946791,0.024571,593.000000,23915.666667,NaN,NaN,0.244261
5,CIC_V2,exact_feature_group,no_graph,observed,3,97096.0,47.000000,0.987607,0.000360,0.990101,...,0.002541,0.000570,0.000129,0.961162,0.002232,1859.000000,17067.000000,NaN,NaN,0.228267
6,UNSW_V2,exact_feature_group,ae_only,observed,3,94027.0,3.000000,0.874737,0.017573,0.969304,...,0.059579,0.008459,0.005360,0.749407,0.041040,3860.333333,4281.666667,0.182224,0.000421,0.018842
7,UNSW_V2,exact_feature_group,full,observed,3,157196.0,51.333333,0.971184,0.001588,0.999288,...,0.001914,0.003752,0.000514,0.984900,0.009738,1711.333333,250.000000,0.174216,0.004835,0.083344
8,UNSW_V2,exact_feature_group,full_shuffled_edges,shuffled,3,157196.0,78.333333,0.969047,0.003720,0.999313,...,0.002988,0.004204,0.000483,0.988088,0.008925,1918.000000,199.333333,0.166569,0.002412,0.111154
9,UNSW_V2,exact_feature_group,graph_classifier,observed,3,112517.0,24.333333,0.969889,0.002599,0.999225,...,0.003698,0.004044,0.000697,0.987366,0.016089,1854.333333,205.666667,0.183109,0.001092,0.045491


=== Paired model differences ===


,dataset,protocol,comparison,n_seeds,delta_f1_macro_mean,delta_f1_macro_std,delta_f1_macro_min,delta_f1_macro_max,f1_wins,delta_auc_roc_mean,delta_pr_auc_mean,delta_pr_auc_std,pr_auc_wins,delta_fp_rate_mean,lower_fpr_wins,delta_tp_rate_mean,higher_tpr_wins,delta_fn_mean,delta_fp_mean
0,CIC_V2,exact_feature_group,full_minus_ae_only,3,0.090742,0.026080,0.063434,0.115391,3,0.051447,0.402260,0.030072,3,-4.340459e-02,3,0.037180,3,-15227.000000,-141863.000000
1,CIC_V2,exact_feature_group,full_minus_full_shuffled_edges,3,0.004645,0.008522,-0.000331,0.014486,1,0.012913,0.018356,0.028178,3,-1.978168e-04,3,0.014042,1,-6719.333333,-648.000000
2,CIC_V2,exact_feature_group,full_minus_graph_classifier,3,0.000653,0.001007,0.000011,0.001814,3,0.000571,0.001348,0.001652,3,-3.922800e-08,2,0.002203,2,-1039.333333,-1.333333
3,CIC_V2,exact_feature_group,full_minus_mlp,3,0.003679,0.008041,-0.001088,0.012962,1,0.004292,0.007073,0.005049,3,-5.668031e-05,2,0.011687,1,-5701.666667,-184.000000
4,CIC_V2,exact_feature_group,full_minus_no_graph,3,0.000138,0.000628,-0.000394,0.000831,1,0.004547,0.007383,0.004542,3,-4.446080e-04,3,-0.002684,0,1147.000000,-1450.000000
5,UNSW_V2,exact_feature_group,full_minus_ae_only,3,0.096448,0.016187,0.087044,0.115139,3,0.029984,0.269812,0.061350,3,-4.707166e-03,3,0.235494,3,-4031.666667,-2149.000000
6,UNSW_V2,exact_feature_group,full_minus_full_shuffled_edges,3,0.002138,0.002133,-0.000043,0.004220,2,-0.000025,-0.001189,0.002935,1,-4.517550e-04,3,-0.003188,1,50.666667,-206.666667
7,UNSW_V2,exact_feature_group,full_minus_graph_classifier,3,0.001296,0.001735,-0.000045,0.003255,2,0.000063,0.002732,0.004604,2,-2.918979e-04,2,-0.002466,1,44.333333,-143.000000
8,UNSW_V2,exact_feature_group,full_minus_mlp,3,0.000612,0.004241,-0.001929,0.005507,1,0.000218,-0.000455,0.002264,1,-4.116104e-06,1,0.001433,2,-32.000000,-2.666667
9,UNSW_V2,exact_feature_group,full_minus_no_graph,3,0.001443,0.003388,-0.001813,0.004949,2,0.000023,-0.001210,0.001465,0,-2.948566e-04,2,-0.002864,1,40.666667,-144.000000


=== Hard-class diagnostic table ===


,dataset,protocol,model,attack,n_seeds,support_mean,recall_mean,recall_std,recall_min,recall_max
0,CIC_V2,exact_feature_group,ae_only,Bot,3,2938.333333,0.489042,0.380977,0.148263,0.900358
1,CIC_V2,exact_feature_group,ae_only,Brute Force -Web,3,453.000000,0.969570,0.049083,0.912946,1.000000
2,CIC_V2,exact_feature_group,ae_only,Brute Force -XSS,3,168.333333,0.974713,0.043799,0.924138,1.000000
3,CIC_V2,exact_feature_group,ae_only,DDoS attacks-LOIC-HTTP,3,70221.333333,0.968356,0.053418,0.906681,0.999987
4,CIC_V2,exact_feature_group,ae_only,Infilteration,3,25364.333333,0.266825,0.256866,0.110305,0.563275
...,...,...,...,...,...,...,...,...,...,...
67,UNSW_V2,exact_feature_group,full,Benign FPR,3,NaN,0.003752,0.000514,NaN,NaN
68,UNSW_V2,exact_feature_group,full_shuffled_edges,Benign FPR,3,NaN,0.004204,0.000483,NaN,NaN
69,UNSW_V2,exact_feature_group,graph_classifier,Benign FPR,3,NaN,0.004044,0.000697,NaN,NaN
70,UNSW_V2,exact_feature_group,mlp,Benign FPR,3,NaN,0.003756,0.000895,NaN,NaN


In [13]:
# Cell 11 — Conservative paper-oriented decisions

def paired_record(dataset_name: str, comparison: str) -> Optional[pd.Series]:
    if paired_summary.empty:
        return None
    subset = paired_summary[
        (paired_summary["dataset"] == dataset_name)
        & (paired_summary["comparison"] == comparison)
    ]
    return None if subset.empty else subset.iloc[0]


def model_record(dataset_name: str, model_name: str) -> Optional[pd.Series]:
    if summary.empty:
        return None
    subset = summary[
        (summary["dataset"] == dataset_name)
        & (summary["model"] == model_name)
    ]
    return None if subset.empty else subset.iloc[0]


decision_rows = []
for dataset_name in PAPER_DATASETS:
    dataset_completeness = completeness_df[
        completeness_df["dataset"] == dataset_name
    ]
    all_tasks_complete = bool(
        len(dataset_completeness) > 0
        and dataset_completeness["completed"].all()
    )

    exact_audit = audit_df[
        (audit_df["dataset"] == dataset_name)
        & (audit_df["protocol"] == "exact_feature_group")
    ] if not audit_df.empty else pd.DataFrame()
    random_audit = audit_df[
        (audit_df["dataset"] == dataset_name)
        & (audit_df["protocol"] == "random")
    ] if not audit_df.empty else pd.DataFrame()

    protocol_valid = bool(
        len(exact_audit) == len(SEEDS)
        and (exact_audit["validation_feature_overlap_rate"] == 0.0).all()
        and (exact_audit["test_feature_overlap_rate"] == 0.0).all()
        and (exact_audit["unique_groups_spanning_all_three"] == 0).all()
    )

    full_vs_no_graph = paired_record(dataset_name, "full_minus_no_graph")
    full_vs_shuffled = paired_record(dataset_name, "full_minus_full_shuffled_edges")
    full_vs_graph_classifier = paired_record(dataset_name, "full_minus_graph_classifier")
    full_vs_ae = paired_record(dataset_name, "full_minus_ae_only")
    no_graph = model_record(dataset_name, "no_graph")
    mlp = model_record(dataset_name, "mlp")

    row = {
        "dataset": dataset_name,
        "paper_tasks_complete": all_tasks_complete,
        "protocol_valid_all_seeds": protocol_valid,
        "random_test_overlap_mean": float(random_audit["test_feature_overlap_rate"].mean()) if not random_audit.empty else np.nan,
        "exact_test_overlap_max": float(exact_audit["test_feature_overlap_rate"].max()) if not exact_audit.empty else np.nan,
    }

    if full_vs_no_graph is not None:
        row.update({
            "full_minus_no_graph_f1_mean": float(full_vs_no_graph["delta_f1_macro_mean"]),
            "full_minus_no_graph_pr_mean": float(full_vs_no_graph["delta_pr_auc_mean"]),
            "full_minus_no_graph_f1_wins": int(full_vs_no_graph["f1_wins"]),
            "full_minus_no_graph_pr_wins": int(full_vs_no_graph["pr_auc_wins"]),
            "full_minus_no_graph_tpr_mean": float(full_vs_no_graph["delta_tp_rate_mean"]),
            "full_minus_no_graph_fpr_mean": float(full_vs_no_graph["delta_fp_rate_mean"]),
        })
    if full_vs_shuffled is not None:
        row.update({
            "full_minus_shuffled_f1_mean": float(full_vs_shuffled["delta_f1_macro_mean"]),
            "full_minus_shuffled_pr_mean": float(full_vs_shuffled["delta_pr_auc_mean"]),
            "full_minus_shuffled_f1_wins": int(full_vs_shuffled["f1_wins"]),
            "full_minus_shuffled_pr_wins": int(full_vs_shuffled["pr_auc_wins"]),
        })
    if full_vs_graph_classifier is not None:
        row["full_minus_graph_classifier_f1_mean"] = float(full_vs_graph_classifier["delta_f1_macro_mean"])
        row["full_minus_graph_classifier_pr_mean"] = float(full_vs_graph_classifier["delta_pr_auc_mean"])
    if full_vs_ae is not None:
        row["full_minus_ae_only_f1_mean"] = float(full_vs_ae["delta_f1_macro_mean"])
    if no_graph is not None and mlp is not None:
        row["no_graph_minus_mlp_f1_mean"] = float(no_graph["f1_macro_mean"] - mlp["f1_macro_mean"])
        row["no_graph_minus_mlp_pr_mean"] = float(no_graph["pr_auc_mean"] - mlp["pr_auc_mean"])

    if not protocol_valid:
        decision = "FIX_SPLIT_IMPLEMENTATION"
        message = "Exact-feature-group overlap is not zero for all paper seeds. Do not report model results."
    elif not all_tasks_complete:
        decision = "PAPER_RUN_INCOMPLETE"
        message = "The split protocol is valid, but one or more expected seed/model tasks are incomplete. Resume before final interpretation."
    else:
        graph_observed = full_vs_no_graph
        graph_randomized = full_vs_shuffled
        observed_positive = bool(
            graph_observed is not None
            and (
                (graph_observed["delta_f1_macro_mean"] > 0 and graph_observed["f1_wins"] >= 2)
                or (graph_observed["delta_pr_auc_mean"] > 0 and graph_observed["pr_auc_wins"] >= 2)
            )
        )
        randomized_positive = bool(
            graph_randomized is not None
            and (
                (graph_randomized["delta_f1_macro_mean"] > 0 and graph_randomized["f1_wins"] >= 2)
                or (graph_randomized["delta_pr_auc_mean"] > 0 and graph_randomized["pr_auc_wins"] >= 2)
            )
        )

        if observed_positive and randomized_positive:
            decision = "GRAPH_SIGNAL_SUPPORTED"
            message = (
                "Observed graph context outperforms No-Graph and the shuffled-edge control on at least one primary ranking/threshold metric in most seeds. "
                "Report this as dataset-dependent graph evidence, not universal superiority."
            )
        elif observed_positive and not randomized_positive:
            decision = "GRAPH_CAPACITY_OR_MIXED_SIGNAL"
            message = (
                "Full improves over No-Graph, but the observed topology does not consistently beat randomized edges. "
                "The gain may reflect graph-model capacity or a metric trade-off rather than reliable topology evidence."
            )
        elif graph_observed is not None and (
            abs(graph_observed["delta_f1_macro_mean"]) <= 0.001
            and abs(graph_observed["delta_pr_auc_mean"]) <= 0.001
        ):
            decision = "CONTROLLED_NEAR_TIE"
            message = (
                "Full and No-Graph remain near tied after duplicate separation. Explicit classification is primary and graph context is auxiliary in this setup."
            )
        else:
            decision = "GRAPH_SIGNAL_NOT_SUPPORTED"
            message = (
                "Full does not consistently improve over No-Graph under the duplicate-disjoint paper protocol. Present it as a negative controlled result."
            )

    row["decision"] = decision
    row["paper_message"] = message
    decision_rows.append(row)

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUTPUT_ROOT / "conference_step4_decision.csv", index=False)
display(decision_df)

lines = [
    "# THAT-IDS Step 4 paper-mode summary",
    "",
    f"- Completed tasks: {completed_count}/{expected_count}",
    f"- Shuffled-edge control enabled: {RUN_SHUFFLED_EDGE_CONTROL}",
    "",
]
for _, row in decision_df.iterrows():
    lines.extend([
        f"## {row['dataset']}",
        f"- Decision: **{row['decision']}**",
        f"- Random test overlap mean: {row['random_test_overlap_mean']:.6f}" if np.isfinite(row['random_test_overlap_mean']) else "- Random test overlap mean: unavailable",
        f"- Exact-group test overlap max: {row['exact_test_overlap_max']:.6f}" if np.isfinite(row['exact_test_overlap_max']) else "- Exact-group test overlap max: unavailable",
        f"- Interpretation: {row['paper_message']}",
        "",
    ])

summary_md_path = OUTPUT_ROOT / "paper_mode_summary.md"
summary_md_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", summary_md_path)


,dataset,paper_tasks_complete,protocol_valid_all_seeds,random_test_overlap_mean,exact_test_overlap_max,full_minus_no_graph_f1_mean,full_minus_no_graph_pr_mean,full_minus_no_graph_f1_wins,full_minus_no_graph_pr_wins,full_minus_no_graph_tpr_mean,...,full_minus_shuffled_pr_mean,full_minus_shuffled_f1_wins,full_minus_shuffled_pr_wins,full_minus_graph_classifier_f1_mean,full_minus_graph_classifier_pr_mean,full_minus_ae_only_f1_mean,no_graph_minus_mlp_f1_mean,no_graph_minus_mlp_pr_mean,decision,paper_message
0,UNSW_V2,True,True,0.934756,0.0,0.001443,-0.001210,2,0,-0.002864,...,-0.001189,2,1,0.001296,0.002732,0.096448,-0.000831,0.000755,GRAPH_SIGNAL_SUPPORTED,Observed graph context outperforms No-Graph an...
1,CIC_V2,True,True,0.548103,0.0,0.000138,0.007383,1,3,-0.002684,...,0.018356,1,3,0.000653,0.001348,0.090742,0.003541,-0.000310,GRAPH_SIGNAL_SUPPORTED,Observed graph context outperforms No-Graph an...


Saved: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_paper_v2/paper/paper_mode_summary.md


In [14]:
# Cell 12 — Package small paper outputs (excludes checkpoints, hashes, predictions)

archive_path = OUTPUT_ROOT / "THAT_IDS_step4_paper_results_bundle.zip"
allowed_suffixes = {".csv", ".json", ".tex", ".md"}

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_ROOT.rglob("*"):
        if not path.is_file() or path == archive_path:
            continue
        if path.suffix.lower() not in allowed_suffixes:
            continue
        # History/config/test metrics are useful; large arrays/checkpoints are excluded by suffix.
        zf.write(path, arcname=str(path.relative_to(OUTPUT_ROOT)))

print("Created result bundle:", archive_path)
print(f"Bundle size: {archive_path.stat().st_size / 1e6:.2f} MB")


Created result bundle: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step4_exact_feature_group_paper_v2/paper/THAT_IDS_step4_paper_results_bundle.zip
Bundle size: 0.35 MB
